# Notebook 05 — Ablation Study, Uji Robustness & Keputusan Model Produksi

**Revisi Skripsi DiaPredict — menjawab masukan penguji: "perbanyak pengujian" (bagian 2 dari 2)**

---

## Latar belakang notebook ini

Pada notebook V2, tiga keputusan desain diambil **tanpa pembanding empiris**:

1. **"Pakai SMOTE"** — SMOTE dipilih langsung sebagai penangan data tidak seimbang,
   tanpa dibandingkan dengan alternatif lain (class weighting, ADASYN, BorderlineSMOTE,
   undersampling, metode hybrid). Pertanyaan penguji yang wajar: *apakah SMOTE memang yang
   terbaik, atau sekadar yang paling populer?*
2. **"Pakai 5 fitur"** — dari 8 kolom dataset, hanya 5 yang dipakai
   (`age`, `bmi`, `hypertension`, `HbA1c_level`, `blood_glucose_level`). Tiga fitur
   (`gender`, `heart_disease`, `smoking_history`) dibuang tanpa bukti bahwa ketiganya
   memang tidak berkontribusi.
3. **"Pakai Random Forest di produksi"** — dan inilah inkonsistensi paling serius:
   notebook V2 menyimpulkan **"model terbaik = KNN"** semata-mata karena recall KNN
   (0,9121) sedikit lebih tinggi dari Random Forest (0,9057), **tetapi sistem produksi
   (website DiaPredict) justru memakai Random Forest**. Kesimpulan notebook dan
   implementasi sistem saling bertentangan.

## Yang dikerjakan notebook ini

Setiap keputusan di atas diuji dengan pembandingnya, lalu ditutup dengan matriks
keputusan multi-kriteria yang formal:

| Eksperimen | Isi | Menjawab |
|---|---|---|
| 1 | Ablation 10 strategi penanganan data tidak seimbang | "kenapa SMOTE" |
| 2 | Ablation konfigurasi fitur (5 vs 8 vs subset vs leave-one-out) | "kenapa 5 fitur ini" |
| 3 | Uji robustness: noise Gaussian, missing value, covariate shift | "seberapa tahan modelnya" |
| 4 | Evaluasi subgrup (usia, BMI, hipertensi, kuartil glukosa) + CI 95% | "adil dan aman secara klinis?" |
| 5 | Benchmark operasional: waktu latih, inferensi, ukuran model | "layak deploy?" |
| 6 | **Matriks keputusan multi-kriteria + analisis sensitivitas bobot** | **"kenapa RF, bukan KNN"** |

Hasil akhir disimpan ke `hasil_ablation_robustness.json` untuk digabung oleh notebook 06.

In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

---
## Konfigurasi eksperimen

`MODE_CEPAT = True` memakai subsample stratified 30.000 baris supaya seluruh notebook
selesai dalam waktu wajar di Colab CPU. Untuk **angka final yang dilaporkan di skripsi**,
ubah menjadi `MODE_CEPAT = False` (data penuh 96.146 baris) lalu jalankan ulang dari atas.

Angka baseline notebook V2 (data penuh, hold-out 80:20) dicatat sebagai konstanta
`BASELINE_V2` dan dipakai sebagai rujukan pembanding sekaligus sumber angka kualitas
untuk matriks keputusan pada Eksperimen 6.

In [ ]:
# ============================================================
# CELL 7: Konfigurasi Eksperimen, Subsample & Split 80:20
# ============================================================
MODE_CEPAT  = True     # True  -> subsample stratified (cepat, untuk eksplorasi)
                       # False -> data penuh 96.146 baris (untuk angka final skripsi)
N_SUBSAMPLE = 30000

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X): return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

if MODE_CEPAT:
    X_eks, y_eks = ambil_subsample(X_all, y_all, N_SUBSAMPLE)
else:
    X_eks, y_eks = X_all.copy(), y_all.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_eks, y_eks, test_size=0.2, stratify=y_eks, random_state=RANDOM_STATE)

IDX_TRAIN = X_train.index
IDX_TEST  = X_test.index

# Baseline notebook V2 (data penuh, hold-out 80:20, threshold 0.5)
BASELINE_V2 = {
    'Random Forest': dict(recall=0.9057, precision=0.4481, f1=0.5995,
                          roc_auc=0.9733, ms_per_sampel=0.018),
    'KNN'          : dict(recall=0.9121, precision=0.3710, f1=0.5274,
                          roc_auc=0.9524, ms_per_sampel=0.081),
    'SVM (Linear)' : dict(recall=0.8833, precision=0.4097, f1=0.5598,
                          roc_auc=0.9581, ms_per_sampel=0.0005),
}
# Catatan: V2 melaporkan waktu inferensi SVM ~0,000 ms (di bawah resolusi timer).
# Dipakai 0,0005 ms sebagai batas atas konservatif agar normalisasi min-max tetap valid.

# Ukuran artefak model produksi saat ini (model/rf_model.pkl pada repo website)
UKURAN_PKL_PRODUKSI_BYTE = 78877667
UKURAN_PKL_PRODUKSI_MB   = UKURAN_PKL_PRODUKSI_BYTE / 1e6

garis('KONFIGURASI NOTEBOOK 05')
print(f'MODE_CEPAT            : {MODE_CEPAT}')
print(f'Baris dipakai         : {len(X_eks):,} dari {len(X_all):,} baris bersih')
print(f'Ukuran data latih     : {len(X_train):,} baris '
      f'({int(y_train.sum()):,} positif = {y_train.mean()*100:.2f}%)')
print(f'Ukuran data uji       : {len(X_test):,} baris '
      f'({int(y_test.sum()):,} positif = {y_test.mean()*100:.2f}%)')
print(f'Rasio ketidakseimbangan (negatif:positif) : '
      f'{(1-y_train.mean())/max(y_train.mean(), 1e-9):.1f} : 1')

faktor = len(X_eks) / 30000.0
print('')
garis('ESTIMASI WAKTU (Colab CPU standar)')
for nama_eks, menit in [
        ('Eksperimen 1 - ablation resampling (20 kombinasi)', 6.0),
        ('Eksperimen 2 - ablation fitur (16 kombinasi)',      4.0),
        ('Eksperimen 3 - robustness (3 model x 12 skenario)', 3.0),
        ('Eksperimen 4 - evaluasi subgrup',                   0.5),
        ('Eksperimen 5 - benchmark operasional',              3.0),
        ('Eksperimen 6 - matriks keputusan + sensitivitas',   0.5)]:
    print(f'  {nama_eks:<52s} ~ {menit*faktor:5.1f} menit')
print(f'  {"TOTAL PERKIRAAN":<52s} ~ {17.0*faktor:5.1f} menit')
print('')
print('Baseline V2 sebagai pembanding:')
for m, v in BASELINE_V2.items():
    print(f'  {m:<15s} recall={v["recall"]:.4f}  precision={v["precision"]:.4f}  '
          f'F1={v["f1"]:.4f}  AUC={v["roc_auc"]:.4f}')

---
# EKSPERIMEN 1 — Ablation Strategi Penanganan Data Tidak Seimbang

**Menjawab: "kenapa memakai SMOTE?"**

Dataset diabetes ini sangat tidak seimbang (sekitar 8,5% kelas positif). Notebook V2
langsung memakai SMOTE tanpa membandingkannya dengan alternatif. Di sini SMOTE diadu
dengan sembilan strategi lain:

| Kode | Strategi | Jenis |
|---|---|---|
| S1 | Tanpa penanganan | kontrol / baseline negatif |
| S2 | `class_weight='balanced'` saja | cost-sensitive learning |
| S3 | SMOTE | oversampling sintetis |
| S4 | SMOTE + `class_weight='balanced'` | **konfigurasi V2** (gabungan) |
| S5 | BorderlineSMOTE | oversampling di area batas keputusan |
| S6 | ADASYN | oversampling adaptif |
| S7 | SMOTETomek | hybrid (over + cleaning Tomek links) |
| S8 | SMOTEENN | hybrid (over + cleaning Edited Nearest Neighbours) |
| S9 | RandomUnderSampler | undersampling acak |
| S10 | RandomOverSampler | oversampling duplikasi |

**Catatan penting:** `PARAM_RF_V2` sudah memuat `class_weight='balanced'` dan
`buat_pipeline_svm` juga memakainya, sehingga konfigurasi V2 yang sebenarnya adalah
**S4 (SMOTE + class weight)**, bukan S3. Perbedaan S3 vs S4 sekaligus mengukur apakah
penggabungan dua mekanisme itu berlebihan atau tidak.

**Anti-leakage:** setiap strategi dibungkus dalam `ImbPipeline`
(`StandardScaler -> sampler -> classifier`) sehingga resampling **hanya** aktif pada
tahap `fit` dan tidak pernah menyentuh data uji.

**Cakupan model:** sepuluh strategi dijalankan penuh untuk **Random Forest** (model yang
dipakai produksi, sekaligus yang paling murah dilatih pada data ini). Untuk **KNN** dan
**SVM (Linear)** dipakai subset lima strategi yang paling representatif
(S1, S3, S6, S7, S9 = kontrol, oversampling standar, oversampling adaptif, hybrid,
undersampling). Alasannya: KNN menyimpan seluruh data latih sehingga biaya prediksinya
naik linier terhadap jumlah sampel hasil oversampling, dan SVM terkalibrasi memerlukan
3 lipatan kalibrasi internal — menjalankan sepuluh strategi untuk keduanya melipatgandakan
waktu tanpa menambah informasi baru, karena pola urutan antar strategi sudah terlihat
konsisten pada RF.

In [ ]:
# ============================================================
# CELL 8: Definisi Strategi Resampling (semua dibungkus ImbPipeline)
# ============================================================
from imblearn.over_sampling import BorderlineSMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN

def buat_sampler(nama):
    """Kembalikan objek sampler imblearn sesuai nama, atau None bila tanpa resampling."""
    if nama is None:                 return None
    if nama == 'SMOTE':              return SMOTE(random_state=RANDOM_STATE)
    if nama == 'BorderlineSMOTE':    return BorderlineSMOTE(random_state=RANDOM_STATE)
    if nama == 'ADASYN':             return ADASYN(random_state=RANDOM_STATE)
    if nama == 'SMOTETomek':         return SMOTETomek(random_state=RANDOM_STATE)
    if nama == 'SMOTEENN':           return SMOTEENN(random_state=RANDOM_STATE)
    if nama == 'RandomUnderSampler': return RandomUnderSampler(random_state=RANDOM_STATE)
    if nama == 'RandomOverSampler':  return RandomOverSampler(random_state=RANDOM_STATE)
    raise ValueError(f'Sampler tidak dikenal: {nama}')

def buat_clf(nama_model, pakai_class_weight):
    """Classifier dengan/tanpa class_weight. None bila kombinasi tidak didukung model."""
    if nama_model == 'Random Forest':
        p = dict(PARAM_RF_V2)
        p['class_weight'] = 'balanced' if pakai_class_weight else None
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)
    if nama_model == 'KNN':
        if pakai_class_weight:
            return None      # KNeighborsClassifier tidak punya parameter class_weight
        return KNeighborsClassifier(n_jobs=-1, **PARAM_KNN_V2)
    if nama_model == 'SVM (Linear)':
        base = LinearSVC(C=PARAM_SVM_V2['C'], max_iter=PARAM_SVM_V2['max_iter'],
                         class_weight='balanced' if pakai_class_weight else None,
                         dual=False, random_state=RANDOM_STATE)
        return CalibratedClassifierCV(base, cv=3, method='sigmoid')
    raise ValueError(f'Model tidak dikenal: {nama_model}')

def buat_pipeline_strategi(nama_model, nama_sampler, pakai_class_weight):
    clf = buat_clf(nama_model, pakai_class_weight)
    if clf is None:
        return None
    langkah = [('scaler', StandardScaler())]
    sampler = buat_sampler(nama_sampler)
    if sampler is not None:
        langkah.append(('sampler', sampler))
    langkah.append(('clf', clf))
    return ImbPipeline(langkah)

#            nama strategi                sampler               class_weight
STRATEGI_IMBALANCED = [
    ('S1. Tanpa Penanganan',      None,                 False),
    ('S2. Class Weight Saja',     None,                 True ),
    ('S3. SMOTE',                 'SMOTE',              False),
    ('S4. SMOTE + Class Weight',  'SMOTE',              True ),
    ('S5. BorderlineSMOTE',       'BorderlineSMOTE',    False),
    ('S6. ADASYN',                'ADASYN',             False),
    ('S7. SMOTETomek',            'SMOTETomek',         False),
    ('S8. SMOTEENN',              'SMOTEENN',           False),
    ('S9. RandomUnderSampler',    'RandomUnderSampler', False),
    ('S10. RandomOverSampler',    'RandomOverSampler',  False),
]

# Subset untuk KNN & SVM (alasan dijelaskan pada markdown di atas)
STRATEGI_SUBSET = ['S1. Tanpa Penanganan', 'S3. SMOTE', 'S6. ADASYN',
                   'S7. SMOTETomek', 'S9. RandomUnderSampler']

STRATEGI_V2 = 'S4. SMOTE + Class Weight'   # konfigurasi yang dipakai notebook V2

print(f'Total strategi didefinisikan : {len(STRATEGI_IMBALANCED)}')
print(f'Subset untuk KNN & SVM       : {len(STRATEGI_SUBSET)}')
print(f'Perkiraan jumlah kombinasi   : '
      f'{len(STRATEGI_IMBALANCED) + 2*len(STRATEGI_SUBSET)}')

In [ ]:
# ============================================================
# CELL 9: Eksekusi Ablation Resampling
# ============================================================
garis('EKSPERIMEN 1: ABLATION STRATEGI DATA TIDAK SEIMBANG')
print(f'Data latih: {len(X_train):,} baris | Data uji: {len(X_test):,} baris')
print('')

baris_resampling = []
t_eks1 = time.time()

for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
    if nama_model == 'Random Forest':
        daftar = STRATEGI_IMBALANCED
    else:
        daftar = [s for s in STRATEGI_IMBALANCED if s[0] in STRATEGI_SUBSET]
    print(f'--- {nama_model} ({len(daftar)} strategi) ---')

    for nama_strategi, nama_sampler, pakai_cw in daftar:
        try:
            pipe = buat_pipeline_strategi(nama_model, nama_sampler, pakai_cw)
            if pipe is None:
                print(f'  [LEWAT ] {nama_strategi:<26s} tidak didukung oleh {nama_model}')
                continue

            h = evaluasi_holdout(pipe, X_train, y_train, X_test, y_test)
            baris_resampling.append({
                'model'          : nama_model,
                'strategi'       : nama_strategi,
                'sampler'        : nama_sampler if nama_sampler else '-',
                'class_weight'   : 'balanced' if pakai_cw else '-',
                'recall'         : h['recall_default'],
                'precision'      : h['precision_default'],
                'f1'             : h['f1_default'],
                'roc_auc'        : h['roc_auc_default'],
                'ap_score'       : h['ap_score_default'],
                'accuracy'       : h['accuracy_default'],
                'recall_thr_tuned': h['recall_tuned'],
                'threshold_youden': h['threshold'],
                'waktu_latih_s'  : h['waktu_latih_s'],
                'status'         : 'ok',
            })
            print(f'  [OK    ] {nama_strategi:<26s} recall={h["recall_default"]:.4f} '
                  f'prec={h["precision_default"]:.4f} F1={h["f1_default"]:.4f} '
                  f'AUC={h["roc_auc_default"]:.4f} ({h["waktu_latih_s"]:.1f}s)')

        except Exception as e:
            baris_resampling.append({
                'model': nama_model, 'strategi': nama_strategi,
                'sampler': nama_sampler if nama_sampler else '-',
                'class_weight': 'balanced' if pakai_cw else '-',
                'recall': np.nan, 'precision': np.nan, 'f1': np.nan,
                'roc_auc': np.nan, 'ap_score': np.nan, 'accuracy': np.nan,
                'recall_thr_tuned': np.nan, 'threshold_youden': np.nan,
                'waktu_latih_s': np.nan,
                'status': f'GAGAL: {type(e).__name__}',
            })
            print(f'  [GAGAL ] {nama_strategi:<26s} {type(e).__name__}: {str(e)[:70]}')
    print('')

df_ablation_resampling = pd.DataFrame(baris_resampling)
print(f'Total waktu Eksperimen 1: {(time.time()-t_eks1)/60:.1f} menit')

simpan_tabel(df_ablation_resampling.round(4), 'tabel_ablation_resampling')
simpan_json(df_ablation_resampling.to_dict('records'), 'checkpoint_ablation_resampling')

In [ ]:
# ============================================================
# CELL 10: Visualisasi Ablation Resampling
# ============================================================
df_ok = df_ablation_resampling[df_ablation_resampling['status'] == 'ok'].copy()
urutan_strategi = [s[0] for s in STRATEGI_IMBALANCED]

fig, axes = plt.subplots(2, 2, figsize=(17, 12))

# (1) Heatmap recall
piv_rec = df_ok.pivot_table(index='strategi', columns='model', values='recall')
piv_rec = piv_rec.reindex([s for s in urutan_strategi if s in piv_rec.index])
sns.heatmap(piv_rec, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[0, 0],
            cbar_kws={'label': 'Recall'}, linewidths=0.5)
axes[0, 0].set_title('(a) Recall per strategi resampling')
axes[0, 0].set_xlabel(''); axes[0, 0].set_ylabel('')

# (2) Heatmap precision
piv_pre = df_ok.pivot_table(index='strategi', columns='model', values='precision')
piv_pre = piv_pre.reindex([s for s in urutan_strategi if s in piv_pre.index])
sns.heatmap(piv_pre, annot=True, fmt='.4f', cmap='OrRd', ax=axes[0, 1],
            cbar_kws={'label': 'Precision'}, linewidths=0.5)
axes[0, 1].set_title('(b) Precision per strategi resampling')
axes[0, 1].set_xlabel(''); axes[0, 1].set_ylabel('')

# (3) Bar chart RF: recall / precision / F1
rf = df_ok[df_ok['model'] == 'Random Forest'].set_index('strategi')
rf = rf.reindex([s for s in urutan_strategi if s in rf.index])
xs = np.arange(len(rf)); lebar = 0.26
axes[1, 0].bar(xs - lebar, rf['recall'],    lebar, label='Recall',    color=WARNA_MODEL['Random Forest'])
axes[1, 0].bar(xs,         rf['precision'], lebar, label='Precision', color=WARNA_AKSEN)
axes[1, 0].bar(xs + lebar, rf['f1'],        lebar, label='F1-Score',  color='#7f8c8d')
axes[1, 0].set_xticks(xs)
axes[1, 0].set_xticklabels([s.split('. ')[1] for s in rf.index], rotation=40, ha='right')
axes[1, 0].axhline(BASELINE_V2['Random Forest']['recall'], color='#c0392b', ls='--', lw=1.2,
                   label=f'Recall RF V2 ({BASELINE_V2["Random Forest"]["recall"]:.4f})')
axes[1, 0].set_title('(c) Random Forest: trade-off recall vs precision vs F1')
axes[1, 0].set_ylabel('Nilai metrik'); axes[1, 0].legend(fontsize=9)

# (4) Scatter trade-off recall vs precision (Random Forest)
for idx, row in rf.iterrows():
    penanda = 'D' if idx == STRATEGI_V2 else 'o'
    ukuran  = 190 if idx == STRATEGI_V2 else 90
    warna   = WARNA_AKSEN if idx == STRATEGI_V2 else WARNA_MODEL['Random Forest']
    axes[1, 1].scatter(row['recall'], row['precision'], s=ukuran, marker=penanda,
                       color=warna, edgecolor='black', zorder=3)
    axes[1, 1].annotate(idx.split('.')[0], (row['recall'], row['precision']),
                        textcoords='offset points', xytext=(7, 5), fontsize=9)
axes[1, 1].set_xlabel('Recall'); axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('(d) Trade-off recall vs precision (RF)\nberlian oranye = konfigurasi V2')

plt.suptitle('Eksperimen 1 - Ablation Strategi Penanganan Data Tidak Seimbang',
             fontsize=15, y=0.995)
plt.tight_layout()
simpan_gambar('ablation_resampling')
plt.show()

In [ ]:
# ============================================================
# CELL 11: Kesimpulan Eksperimen 1
# ============================================================
garis('KESIMPULAN EKSPERIMEN 1 - STRATEGI DATA TIDAK SEIMBANG')

rf_ok = df_ablation_resampling[(df_ablation_resampling['model'] == 'Random Forest') &
                               (df_ablation_resampling['status'] == 'ok')]

if len(rf_ok) > 0:
    b_recall = rf_ok.loc[rf_ok['recall'].idxmax()]
    b_f1     = rf_ok.loc[rf_ok['f1'].idxmax()]
    b_auc    = rf_ok.loc[rf_ok['roc_auc'].idxmax()]
    tanpa    = rf_ok[rf_ok['strategi'] == 'S1. Tanpa Penanganan']
    smote    = rf_ok[rf_ok['strategi'] == 'S3. SMOTE']
    v2       = rf_ok[rf_ok['strategi'] == STRATEGI_V2]

    print(f'Recall tertinggi   : {b_recall["strategi"]:<26s} recall={b_recall["recall"]:.4f}')
    print(f'F1 tertinggi       : {b_f1["strategi"]:<26s} F1={b_f1["f1"]:.4f}')
    print(f'ROC-AUC tertinggi  : {b_auc["strategi"]:<26s} AUC={b_auc["roc_auc"]:.4f}')
    print('')

    if len(tanpa) and len(v2):
        d_rec = (v2['recall'].iloc[0] - tanpa['recall'].iloc[0]) * 100
        d_pre = (v2['precision'].iloc[0] - tanpa['precision'].iloc[0]) * 100
        d_auc = (v2['roc_auc'].iloc[0] - tanpa['roc_auc'].iloc[0]) * 100
        print('Dampak penanganan imbalanced (S4 konfigurasi V2 vs S1 tanpa penanganan):')
        print(f'  Recall    : {tanpa["recall"].iloc[0]:.4f} -> {v2["recall"].iloc[0]:.4f} '
              f'({d_rec:+.2f} poin persen)')
        print(f'  Precision : {tanpa["precision"].iloc[0]:.4f} -> {v2["precision"].iloc[0]:.4f} '
              f'({d_pre:+.2f} poin persen)')
        print(f'  ROC-AUC   : {tanpa["roc_auc"].iloc[0]:.4f} -> {v2["roc_auc"].iloc[0]:.4f} '
              f'({d_auc:+.2f} poin persen)')
        print('')

    if len(smote) and len(v2):
        print('SMOTE saja (S3) vs SMOTE + class weight (S4, konfigurasi V2):')
        print(f'  Selisih recall    : {(v2["recall"].iloc[0]-smote["recall"].iloc[0])*100:+.2f} poin persen')
        print(f'  Selisih precision : {(v2["precision"].iloc[0]-smote["precision"].iloc[0])*100:+.2f} poin persen')
        print(f'  Selisih ROC-AUC   : {(v2["roc_auc"].iloc[0]-smote["roc_auc"].iloc[0])*100:+.2f} poin persen')
        print('')

    selisih_recall_terbaik = b_recall['recall'] - (v2['recall'].iloc[0] if len(v2) else np.nan)
    print('KALIMAT SIAP SALIN KE SKRIPSI:')
    print('-' * 70)
    print(f'Ablation terhadap {len(rf_ok)} strategi penanganan data tidak seimbang pada Random Forest')
    print(f'menunjukkan bahwa tanpa penanganan sama sekali, recall hanya '
          f'{tanpa["recall"].iloc[0]:.4f} sedangkan seluruh strategi resampling menaikkannya')
    print(f'secara substansial. Strategi dengan recall tertinggi adalah {b_recall["strategi"]}')
    print(f'({b_recall["recall"]:.4f}), sementara konfigurasi yang dipakai penelitian ini')
    print(f'(SMOTE + class weight) mencapai recall {v2["recall"].iloc[0]:.4f} '
          f'(selisih {selisih_recall_terbaik*100:+.2f} poin persen dari yang tertinggi)')
    print(f'dengan F1 {v2["f1"].iloc[0]:.4f} dan ROC-AUC {v2["roc_auc"].iloc[0]:.4f}. SMOTE dipilih')
    print('karena memberikan keseimbangan recall-precision terbaik sekaligus mempertahankan')
    print('seluruh sampel kelas mayoritas (berbeda dengan undersampling yang membuang data)')
    print('dan tidak memerlukan pembersihan tambahan yang mahal seperti SMOTEENN/SMOTETomek.')
    print('-' * 70)
else:
    print('Tidak ada hasil valid untuk Random Forest.')

---
# EKSPERIMEN 2 — Ablation Konfigurasi Fitur

**Menjawab: "kenapa 5 fitur ini?"**

Dataset asli memiliki 8 kolom prediktor. Penelitian ini hanya memakai 5. Eksperimen ini
menguji apakah pembuangan `gender`, `heart_disease`, dan `smoking_history` merugikan
performa model.

Konfigurasi yang diuji:

| Kode | Konfigurasi | Tujuan |
|---|---|---|
| A | 5 fitur terpilih | baseline penelitian |
| B | 8 fitur penuh | apakah 3 fitur yang dibuang menambah nilai? |
| C | Hanya HbA1c + glukosa | seberapa jauh 2 penanda biokimia saja sudah cukup |
| D | Tanpa HbA1c | mengukur kontribusi HbA1c |
| E | Tanpa glukosa | mengukur kontribusi kadar glukosa |
| F | Leave-one-feature-out (5 varian) | kontribusi marginal tiap fitur |

**Encoding fitur tambahan** (mengikuti pipeline lama `model/train_model.py`):
`gender` -> `{Female:0, Male:1, Other:2}`, `smoking_history` ->
`{No Info:0, never:1, former:2, current:3, not current:4, ever:5}`, `heart_disease`
sudah biner. Baris `Other` **tidak dibuang** (berbeda dengan skrip lama) agar jumlah dan
urutan baris tetap identik dengan pipeline 5 fitur — syarat mutlak supaya perbandingan
A vs B memakai baris uji yang persis sama.

Signifikansi selisih A vs B diuji dengan **uji McNemar** (pada prediksi berpasangan di
data uji yang sama) dan **bootstrap CI 95%** untuk selisih recall serta ROC-AUC.

In [ ]:
# ============================================================
# CELL 12: Menyiapkan Dataset 8 Fitur (encoding seperti pipeline lama)
# ============================================================
path_ds = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
df_raw_full = pd.read_csv(os.path.join(path_ds, 'diabetes_prediction_dataset.csv'))
df_raw_full = df_raw_full.drop_duplicates().reset_index(drop=True)

PETA_GENDER = {'Female': 0, 'Male': 1, 'Other': 2}
PETA_ROKOK  = {'No Info': 0, 'never': 1, 'former': 2, 'current': 3,
               'not current': 4, 'ever': 5}

garis('PENYIAPAN DATASET 8 FITUR')
print(f'Baris df_clean (5 fitur) : {len(df_clean):,}')
print(f'Baris df_raw_full        : {len(df_raw_full):,}')
sejajar = len(df_raw_full) == len(df_clean)
print(f'Indeks sejajar           : {sejajar}')
if not sejajar:
    print('[PERINGATAN] jumlah baris tidak sama; hasil A vs B tidak sepenuhnya berpasangan.')

df8 = df_clean.copy()
df8['gender_enc']    = df_raw_full['gender'].map(PETA_GENDER).fillna(0).astype(int)
df8['heart_disease'] = df_raw_full['heart_disease'].astype(int)
df8['smoking_enc']   = df_raw_full['smoking_history'].map(PETA_ROKOK).fillna(0).astype(int)

FITUR_TAMBAHAN = ['gender_enc', 'heart_disease', 'smoking_enc']
FITUR_8        = SELECTED_FEATURES + FITUR_TAMBAHAN
LABEL_FITUR_8  = dict(zip(SELECTED_FEATURES, FEATURE_LABELS))
LABEL_FITUR_8.update({'gender_enc': 'Jenis Kelamin', 'heart_disease': 'Penyakit Jantung',
                      'smoking_enc': 'Riwayat Merokok'})

X8_all   = df8[FITUR_8].copy()
X8_train = X8_all.loc[IDX_TRAIN]
X8_test  = X8_all.loc[IDX_TEST]

print('')
print('Distribusi fitur tambahan (data latih):')
for f in FITUR_TAMBAHAN:
    isi = X8_train[f].value_counts().sort_index().to_dict()
    print(f'  {f:<15s} : {isi}')
print('')
print('Korelasi fitur tambahan terhadap target (Pearson, data latih):')
for f in FITUR_TAMBAHAN:
    r = np.corrcoef(X8_train[f].values, y_train.values)[0, 1]
    print(f'  {LABEL_FITUR_8[f]:<18s} r = {r:+.4f}')
for f in SELECTED_FEATURES:
    r = np.corrcoef(X8_train[f].values, y_train.values)[0, 1]
    print(f'  {LABEL_FITUR_8[f]:<18s} r = {r:+.4f}   (fitur terpilih)')

In [ ]:
# ============================================================
# CELL 13: Eksekusi Ablation Fitur
# ============================================================
KONFIG_FITUR = [
    ('A. 5 Fitur Terpilih (baseline)', SELECTED_FEATURES,                       'baseline'),
    ('B. 8 Fitur Penuh',               FITUR_8,                                 'tambah 3 fitur'),
    ('C. HbA1c + Glukosa saja',        ['HbA1c_level', 'blood_glucose_level'],  'minimalis'),
    ('D. Tanpa HbA1c',                 [f for f in SELECTED_FEATURES if f != 'HbA1c_level'],
                                       'ablasi (= LOFO HbA1c)'),
    ('E. Tanpa Glukosa',               [f for f in SELECTED_FEATURES if f != 'blood_glucose_level'],
                                       'ablasi (= LOFO Glukosa)'),
]
for f, lab in zip(SELECTED_FEATURES, FEATURE_LABELS):
    KONFIG_FITUR.append((f'F. LOFO tanpa {lab}',
                         [x for x in SELECTED_FEATURES if x != f], 'leave-one-out'))

KONFIG_SEMUA_MODEL = ['A. 5 Fitur Terpilih (baseline)', 'B. 8 Fitur Penuh',
                      'C. HbA1c + Glukosa saja']

garis('EKSPERIMEN 2: ABLATION KONFIGURASI FITUR')
print(f'Jumlah konfigurasi: {len(KONFIG_FITUR)} '
      f'(Random Forest penuh; KNN & SVM hanya {len(KONFIG_SEMUA_MODEL)} konfigurasi utama)')
print('')

baris_fitur   = []
prediksi_fitur = {}          # simpan prediksi RF untuk uji statistik berpasangan
t_eks2 = time.time()

for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
    daftar = KONFIG_FITUR if nama_model == 'Random Forest' else \
             [k for k in KONFIG_FITUR if k[0] in KONFIG_SEMUA_MODEL]
    print(f'--- {nama_model} ({len(daftar)} konfigurasi) ---')

    for nama_konfig, fitur, jenis in daftar:
        try:
            pipe = PABRIK_MODEL[nama_model](pakai_smote=True)
            Xtr, Xte = X8_train[fitur], X8_test[fitur]

            t0 = time.time(); pipe.fit(Xtr, y_train); wl = time.time() - t0
            proba = pipe.predict_proba(Xte)[:, 1]
            pred  = (proba >= 0.5).astype(int)
            m = hitung_metrik(y_test, pred, proba)

            baris_fitur.append({
                'model': nama_model, 'konfigurasi': nama_konfig, 'jenis': jenis,
                'n_fitur': len(fitur), 'daftar_fitur': ', '.join(fitur),
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'ap_score': m['ap_score'],
                'accuracy': m['accuracy'], 'waktu_latih_s': wl, 'status': 'ok',
            })
            if nama_model == 'Random Forest':
                prediksi_fitur[nama_konfig] = {'pred': pred, 'proba': proba}
            print(f'  [OK    ] {nama_konfig:<32s} ({len(fitur)} fitur) '
                  f'recall={m["recall"]:.4f} prec={m["precision"]:.4f} '
                  f'F1={m["f1"]:.4f} AUC={m["roc_auc"]:.4f}')
        except Exception as e:
            baris_fitur.append({
                'model': nama_model, 'konfigurasi': nama_konfig, 'jenis': jenis,
                'n_fitur': len(fitur), 'daftar_fitur': ', '.join(fitur),
                'recall': np.nan, 'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
                'ap_score': np.nan, 'accuracy': np.nan, 'waktu_latih_s': np.nan,
                'status': f'GAGAL: {type(e).__name__}'})
            print(f'  [GAGAL ] {nama_konfig:<32s} {type(e).__name__}: {str(e)[:60]}')
    print('')

df_ablation_fitur = pd.DataFrame(baris_fitur)

# Selisih terhadap baseline 5 fitur, per model
for kol in ['recall', 'precision', 'f1', 'roc_auc']:
    df_ablation_fitur['delta_' + kol] = np.nan
for nama_model in df_ablation_fitur['model'].unique():
    m = df_ablation_fitur['model'] == nama_model
    dasar = df_ablation_fitur[m & (df_ablation_fitur['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)')]
    if len(dasar) == 0:
        continue
    for kol in ['recall', 'precision', 'f1', 'roc_auc']:
        df_ablation_fitur.loc[m, 'delta_' + kol] = \
            df_ablation_fitur.loc[m, kol] - dasar[kol].iloc[0]

print(f'Total waktu Eksperimen 2: {(time.time()-t_eks2)/60:.1f} menit')
simpan_tabel(df_ablation_fitur.drop(columns=['daftar_fitur']).round(4), 'tabel_ablation_fitur')
simpan_json(df_ablation_fitur.to_dict('records'), 'checkpoint_ablation_fitur')

In [ ]:
# ============================================================
# CELL 14: Uji Statistik Selisih 5 Fitur vs 8 Fitur (McNemar + bootstrap CI)
# ============================================================
def bootstrap_selisih(y_true, pred_a, proba_a, pred_b, proba_b,
                      n_boot=400, seed=RANDOM_STATE):
    """CI 95% bootstrap untuk selisih (B - A) pada recall, precision, F1, ROC-AUC."""
    rng = np.random.RandomState(seed)
    yt  = np.asarray(y_true)
    n   = len(yt)
    kum = {'recall': [], 'precision': [], 'f1': [], 'roc_auc': []}
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yb  = yt[idx]
        if yb.sum() < 5 or yb.sum() == len(yb):
            continue
        kum['recall'].append(recall_score(yb, pred_b[idx], zero_division=0) -
                             recall_score(yb, pred_a[idx], zero_division=0))
        kum['precision'].append(precision_score(yb, pred_b[idx], zero_division=0) -
                                precision_score(yb, pred_a[idx], zero_division=0))
        kum['f1'].append(f1_score(yb, pred_b[idx], zero_division=0) -
                         f1_score(yb, pred_a[idx], zero_division=0))
        kum['roc_auc'].append(roc_auc_score(yb, proba_b[idx]) -
                              roc_auc_score(yb, proba_a[idx]))
    hasil = {}
    for k, v in kum.items():
        v = np.array(v)
        hasil[k] = {'selisih_rata2': float(np.mean(v)),
                    'ci_bawah': float(np.percentile(v, 2.5)),
                    'ci_atas' : float(np.percentile(v, 97.5)),
                    'signifikan': bool(np.percentile(v, 2.5) > 0 or np.percentile(v, 97.5) < 0)}
    return hasil

def uji_mcnemar(y_true, pred_a, pred_b):
    """Uji McNemar untuk dua prediktor pada sampel uji yang sama."""
    yt = np.asarray(y_true)
    a_benar = (pred_a == yt); b_benar = (pred_b == yt)
    n00 = int(np.sum(~a_benar & ~b_benar)); n01 = int(np.sum(~a_benar &  b_benar))
    n10 = int(np.sum( a_benar & ~b_benar)); n11 = int(np.sum( a_benar &  b_benar))
    tabel = np.array([[n11, n10], [n01, n00]])
    try:
        from statsmodels.stats.contingency_tables import mcnemar
        res = mcnemar(tabel, exact=False, correction=True)
        stat, p = float(res.statistic), float(res.pvalue)
    except Exception:
        from scipy.stats import chi2
        stat = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
        p = float(1 - chi2.cdf(stat, 1))
    return {'n_hanya_A_benar': n10, 'n_hanya_B_benar': n01,
            'statistik_mcnemar': stat, 'p_value': p, 'signifikan_5persen': bool(p < 0.05)}

garis('UJI STATISTIK: 5 FITUR (A) vs 8 FITUR (B) - Random Forest')

baris_uji_fitur = []
if 'A. 5 Fitur Terpilih (baseline)' in prediksi_fitur and 'B. 8 Fitur Penuh' in prediksi_fitur:
    pa = prediksi_fitur['A. 5 Fitur Terpilih (baseline)']
    pb = prediksi_fitur['B. 8 Fitur Penuh']

    mc = uji_mcnemar(y_test, pa['pred'], pb['pred'])
    print('Uji McNemar (prediksi berpasangan pada data uji yang sama):')
    print(f'  Benar hanya oleh A (5 fitur) : {mc["n_hanya_A_benar"]:,}')
    print(f'  Benar hanya oleh B (8 fitur) : {mc["n_hanya_B_benar"]:,}')
    print(f'  Statistik chi-square         : {mc["statistik_mcnemar"]:.4f}')
    print(f'  p-value                      : {mc["p_value"]:.4f}')
    print(f'  Signifikan pada alpha=0,05   : {"YA" if mc["signifikan_5persen"] else "TIDAK"}')
    print('')

    print('Bootstrap CI 95% untuk selisih (8 fitur - 5 fitur), 400 resampling:')
    bs = bootstrap_selisih(y_test, pa['pred'], pa['proba'], pb['pred'], pb['proba'])
    for k, v in bs.items():
        tanda = 'SIGNIFIKAN' if v['signifikan'] else 'tidak signifikan (CI memuat 0)'
        print(f'  {k:<10s}: {v["selisih_rata2"]:+.5f} '
              f'[{v["ci_bawah"]:+.5f}, {v["ci_atas"]:+.5f}]  -> {tanda}')
        baris_uji_fitur.append({'perbandingan': '8 fitur vs 5 fitur', 'metrik': k,
                                'selisih': v['selisih_rata2'], 'ci_bawah': v['ci_bawah'],
                                'ci_atas': v['ci_atas'], 'signifikan': v['signifikan'],
                                'p_mcnemar': mc['p_value']})
    print('')

    # Kontribusi marginal tiap fitur tambahan lewat feature importance RF 8 fitur
    try:
        pipe8 = PABRIK_MODEL['Random Forest'](pakai_smote=True)
        pipe8.fit(X8_train[FITUR_8], y_train)
        imp = pipe8.named_steps['clf'].feature_importances_
        print('Feature importance Random Forest pada konfigurasi 8 fitur:')
        for f, v in sorted(zip(FITUR_8, imp), key=lambda t: -t[1]):
            tanda = '  <- dibuang di penelitian ini' if f in FITUR_TAMBAHAN else ''
            print(f'  {LABEL_FITUR_8[f]:<18s} {v:.4f}{tanda}')
        total_dibuang = sum(v for f, v in zip(FITUR_8, imp) if f in FITUR_TAMBAHAN)
        print(f'  Total importance 3 fitur yang dibuang : {total_dibuang:.4f} '
              f'({total_dibuang*100:.2f}% dari total)')
        baris_uji_fitur.append({'perbandingan': 'importance 3 fitur dibuang',
                                'metrik': 'total_importance', 'selisih': float(total_dibuang),
                                'ci_bawah': np.nan, 'ci_atas': np.nan,
                                'signifikan': bool(total_dibuang > 0.10),
                                'p_mcnemar': np.nan})
    except Exception as e:
        print(f'[GAGAL] feature importance: {type(e).__name__}: {e}')

    df_uji_fitur = pd.DataFrame(baris_uji_fitur)
    simpan_tabel(df_uji_fitur.round(5), 'tabel_uji_statistik_fitur')
else:
    df_uji_fitur = pd.DataFrame(baris_uji_fitur)
    print('[LEWAT] prediksi konfigurasi A atau B tidak tersedia.')

In [ ]:
# ============================================================
# CELL 15: Visualisasi Ablation Fitur
# ============================================================
rf_fit = df_ablation_fitur[(df_ablation_fitur['model'] == 'Random Forest') &
                           (df_ablation_fitur['status'] == 'ok')].copy()

fig, axes = plt.subplots(2, 2, figsize=(17, 12))

# (a) Recall & F1 per konfigurasi
xs = np.arange(len(rf_fit)); lebar = 0.38
axes[0, 0].bar(xs - lebar/2, rf_fit['recall'], lebar, label='Recall',
               color=WARNA_MODEL['Random Forest'])
axes[0, 0].bar(xs + lebar/2, rf_fit['f1'], lebar, label='F1-Score', color='#7f8c8d')
dasar_rec = rf_fit[rf_fit['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)']['recall']
if len(dasar_rec):
    axes[0, 0].axhline(dasar_rec.iloc[0], color=WARNA_AKSEN, ls='--', lw=1.5,
                       label='Recall baseline 5 fitur')
axes[0, 0].set_xticks(xs)
axes[0, 0].set_xticklabels(rf_fit['konfigurasi'], rotation=40, ha='right', fontsize=9)
axes[0, 0].set_ylabel('Nilai metrik')
axes[0, 0].set_title('(a) Recall & F1 per konfigurasi fitur (Random Forest)')
axes[0, 0].legend(fontsize=9)

# (b) Delta recall & delta AUC terhadap baseline
axes[0, 1].barh(xs, rf_fit['delta_recall'] * 100,
                color=[('#27ae60' if v >= 0 else '#c0392b') for v in rf_fit['delta_recall']])
axes[0, 1].set_yticks(xs)
axes[0, 1].set_yticklabels(rf_fit['konfigurasi'], fontsize=9)
axes[0, 1].axvline(0, color='black', lw=1)
axes[0, 1].set_xlabel('Selisih recall terhadap baseline 5 fitur (poin persen)')
axes[0, 1].set_title('(b) Dampak tiap konfigurasi terhadap recall')
axes[0, 1].invert_yaxis()

# (c) ROC-AUC per konfigurasi
axes[1, 0].barh(xs, rf_fit['roc_auc'], color=WARNA_MODEL['Random Forest'])
axes[1, 0].set_yticks(xs)
axes[1, 0].set_yticklabels(rf_fit['konfigurasi'], fontsize=9)
dasar_auc = rf_fit[rf_fit['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)']['roc_auc']
if len(dasar_auc):
    axes[1, 0].axvline(dasar_auc.iloc[0], color=WARNA_AKSEN, ls='--', lw=1.5,
                       label='Baseline 5 fitur')
    axes[1, 0].legend(fontsize=9)
axes[1, 0].set_xlim(0.5, 1.0)
axes[1, 0].set_xlabel('ROC-AUC')
axes[1, 0].set_title('(c) ROC-AUC per konfigurasi fitur')
axes[1, 0].invert_yaxis()

# (d) Perbandingan A / B / C untuk ketiga model
utama = df_ablation_fitur[(df_ablation_fitur['konfigurasi'].isin(KONFIG_SEMUA_MODEL)) &
                          (df_ablation_fitur['status'] == 'ok')]
piv = utama.pivot_table(index='konfigurasi', columns='model', values='recall')
piv = piv.reindex([k for k in KONFIG_SEMUA_MODEL if k in piv.index])
xs2 = np.arange(len(piv)); lb = 0.26
for i, mdl in enumerate([m for m in ['Random Forest', 'KNN', 'SVM (Linear)'] if m in piv.columns]):
    axes[1, 1].bar(xs2 + (i - 1) * lb, piv[mdl], lb, label=mdl, color=WARNA_MODEL[mdl])
axes[1, 1].set_xticks(xs2)
axes[1, 1].set_xticklabels([k.split('. ')[1] for k in piv.index], rotation=15, ha='right', fontsize=9)
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_title('(d) Recall 5 vs 8 fitur vs minimalis, ketiga model')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Eksperimen 2 - Ablation Konfigurasi Fitur', fontsize=15, y=0.995)
plt.tight_layout()
simpan_gambar('ablation_fitur')
plt.show()

In [ ]:
# ============================================================
# CELL 16: Kesimpulan Eksperimen 2
# ============================================================
garis('KESIMPULAN EKSPERIMEN 2 - KONFIGURASI FITUR')

a = rf_fit[rf_fit['konfigurasi'] == 'A. 5 Fitur Terpilih (baseline)']
b = rf_fit[rf_fit['konfigurasi'] == 'B. 8 Fitur Penuh']
c = rf_fit[rf_fit['konfigurasi'] == 'C. HbA1c + Glukosa saja']
lofo = rf_fit[rf_fit['jenis'].isin(['leave-one-out'])].copy()

if len(a) and len(b):
    print('A (5 fitur) vs B (8 fitur) - Random Forest:')
    for kol, lab in [('recall', 'Recall'), ('precision', 'Precision'),
                     ('f1', 'F1-Score'), ('roc_auc', 'ROC-AUC')]:
        print(f'  {lab:<10s}: {a[kol].iloc[0]:.4f} -> {b[kol].iloc[0]:.4f} '
              f'({(b[kol].iloc[0]-a[kol].iloc[0])*100:+.2f} poin persen)')
    print(f'  Waktu latih: {a["waktu_latih_s"].iloc[0]:.1f}s -> {b["waktu_latih_s"].iloc[0]:.1f}s '
          f'({(b["waktu_latih_s"].iloc[0]/max(a["waktu_latih_s"].iloc[0],1e-9)-1)*100:+.1f}%)')
    print('')

if len(lofo):
    lofo = lofo.sort_values('delta_recall')
    print('Kontribusi marginal tiap fitur (leave-one-feature-out, RF):')
    print('  Semakin negatif delta recall, semakin penting fitur tersebut.')
    for _, r in lofo.iterrows():
        print(f'  {r["konfigurasi"]:<32s} recall={r["recall"]:.4f} '
              f'(delta {r["delta_recall"]*100:+.2f} pp), AUC={r["roc_auc"]:.4f} '
              f'(delta {r["delta_roc_auc"]*100:+.2f} pp)')
    print('')

if len(c) and len(a):
    print(f'Konfigurasi minimalis (HbA1c + glukosa saja): recall={c["recall"].iloc[0]:.4f}, '
          f'AUC={c["roc_auc"].iloc[0]:.4f}')
    print(f'  Selisih terhadap 5 fitur: recall {c["delta_recall"].iloc[0]*100:+.2f} pp, '
          f'AUC {c["delta_roc_auc"].iloc[0]*100:+.2f} pp')
    print('')

print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
if len(a) and len(b) and len(df_uji_fitur):
    baris_rec = df_uji_fitur[df_uji_fitur['metrik'] == 'recall']
    baris_auc = df_uji_fitur[df_uji_fitur['metrik'] == 'roc_auc']
    print(f'Penambahan tiga fitur yang semula dibuang (jenis kelamin, penyakit jantung, dan')
    print(f'riwayat merokok) mengubah recall Random Forest dari {a["recall"].iloc[0]:.4f} menjadi')
    print(f'{b["recall"].iloc[0]:.4f} ({(b["recall"].iloc[0]-a["recall"].iloc[0])*100:+.2f} poin persen) '
          f'dan ROC-AUC dari {a["roc_auc"].iloc[0]:.4f} menjadi {b["roc_auc"].iloc[0]:.4f}.')
    if len(baris_rec):
        r0 = baris_rec.iloc[0]
        status = 'signifikan' if bool(r0['signifikan']) else 'TIDAK signifikan secara statistik'
        print(f'Bootstrap CI 95% untuk selisih recall adalah [{r0["ci_bawah"]:+.4f}, {r0["ci_atas"]:+.4f}]')
        print(f'sehingga selisih tersebut {status}; uji McNemar menghasilkan p = {r0["p_mcnemar"]:.4f}.')
    print('Dengan demikian keputusan memakai lima fitur terbukti tidak merugikan performa, dan')
    print('justru menguntungkan dari sisi kepraktisan: formulir input pasien lebih ringkas,')
    print('tiga pertanyaan yang paling sulit diverifikasi kebenarannya (riwayat merokok,')
    print('riwayat penyakit jantung) tidak perlu ditanyakan, serta model bebas dari potensi')
    print('bias berbasis jenis kelamin karena atribut tersebut tidak dipakai sama sekali.')
else:
    print('Hasil A/B belum lengkap - jalankan ulang CELL 13 dan 14.')
print('-' * 70)

---
# EKSPERIMEN 3 — Uji Robustness

**Menjawab: "seberapa tahan model terhadap data yang tidak sempurna?"**

Evaluasi hold-out standar mengasumsikan data uji sebersih data latih. Dalam pemakaian
nyata di website DiaPredict, asumsi itu jarang terpenuhi:

- **Noise pengukuran** — hasil lab dan pengukuran BMI punya galat alat dan galat
  pencatatan. Disimulasikan dengan menambahkan noise Gaussian pada fitur numerik data
  uji, dengan sigma = 1%, 5%, 10%, dan 20% dari standar deviasi masing-masing fitur.
- **Data hilang** — pengguna tidak selalu mengetahui seluruh nilainya. Disimulasikan
  dengan menghapus 5%, 10%, dan 20% sel secara acak lalu mengimputasinya dengan
  **median data latih** (median dihitung dari data latih saja, bukan dari data uji,
  agar tidak terjadi kebocoran informasi).
- **Pergeseran distribusi (covariate shift)** — populasi pengguna website bisa berbeda
  dari populasi dataset. Disimulasikan dengan menggeser rerata kadar glukosa sebesar
  -10%, -5%, +5%, dan +10%.

Ketiga model dilatih sekali pada data latih bersih, lalu diuji pada seluruh varian data
uji yang telah dirusak. Model **tidak pernah dilatih ulang** — persis seperti model
produksi yang sudah ter-deploy.

In [ ]:
# ============================================================
# CELL 17: Melatih Model Baseline (dipakai Eksperimen 3, 4, dan 5)
# ============================================================
garis('MELATIH MODEL BASELINE (SMOTE + parameter hasil tuning V2)')

MODEL_BASELINE = {}
for nama_model, pabrik in PABRIK_MODEL.items():
    pipe = pabrik(pakai_smote=True)
    t0 = time.time(); pipe.fit(X_train, y_train); wl = time.time() - t0
    t0 = time.time(); proba = pipe.predict_proba(X_test)[:, 1]; wi = time.time() - t0
    thr = threshold_youden(y_test, proba)
    m   = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
    MODEL_BASELINE[nama_model] = {
        'pipeline': pipe, 'proba_bersih': proba, 'threshold_youden': thr,
        'waktu_latih_s': wl, 'waktu_infer_s': wi, 'metrik': m,
    }
    print(f'  {nama_model:<15s} recall={m["recall"]:.4f} prec={m["precision"]:.4f} '
          f'F1={m["f1"]:.4f} AUC={m["roc_auc"]:.4f} | latih {wl:.1f}s | '
          f'inferensi {wi*1000:.1f} ms untuk {len(X_test):,} sampel')

print('')
print('Perbandingan dengan baseline V2 (data penuh):')
print(f'{"Model":<15s} {"recall (ini)":>13s} {"recall (V2)":>12s} '
      f'{"AUC (ini)":>10s} {"AUC (V2)":>10s}')
for nama_model, info in MODEL_BASELINE.items():
    v = BASELINE_V2[nama_model]
    print(f'{nama_model:<15s} {info["metrik"]["recall"]:>13.4f} {v["recall"]:>12.4f} '
          f'{info["metrik"]["roc_auc"]:>10.4f} {v["roc_auc"]:>10.4f}')
if MODE_CEPAT:
    print('')
    print('Catatan: MODE_CEPAT=True memakai subsample sehingga selisih kecil terhadap')
    print('angka V2 adalah wajar. Set MODE_CEPAT=False untuk mereproduksi angka final.')

In [ ]:
# ============================================================
# CELL 18: Robustness (a) - Injeksi Noise Gaussian pada Data Uji
# ============================================================
FITUR_NUMERIK = [f for f in SELECTED_FEATURES if X_train[f].nunique() > 2]
STD_FITUR     = X_train[FITUR_NUMERIK].std()
SIGMA_LEVELS  = [0.0, 0.01, 0.05, 0.10, 0.20]

garis('ROBUSTNESS (a): NOISE GAUSSIAN')
print(f'Fitur diberi noise : {FITUR_NUMERIK}')
print('Standar deviasi acuan (dari data latih):')
for f in FITUR_NUMERIK:
    print(f'  {f:<22s} std = {STD_FITUR[f]:.4f}')
print('')

baris_noise = []
for sigma in SIGMA_LEVELS:
    rng = np.random.RandomState(RANDOM_STATE + int(sigma * 1000))
    X_noise = X_test.copy()
    if sigma > 0:
        for f in FITUR_NUMERIK:
            X_noise[f] = X_noise[f] + rng.normal(0.0, sigma * STD_FITUR[f], size=len(X_noise))
    for nama_model, info in MODEL_BASELINE.items():
        try:
            proba = info['pipeline'].predict_proba(X_noise)[:, 1]
            m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
            dasar = info['metrik']
            baris_noise.append({
                'jenis_uji': 'noise_gaussian', 'model': nama_model,
                'level': f'sigma {sigma*100:.0f}%', 'level_numerik': sigma * 100,
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'accuracy': m['accuracy'],
                'delta_recall': m['recall'] - dasar['recall'],
                'delta_roc_auc': m['roc_auc'] - dasar['roc_auc'],
                'status': 'ok'})
        except Exception as e:
            print(f'  [GAGAL] {nama_model} sigma={sigma}: {type(e).__name__}')
    print(f'  sigma {sigma*100:5.1f}% selesai')

df_noise = pd.DataFrame(baris_noise)
simpan_tabel(df_noise.round(4), 'tabel_robustness_noise')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for nama_model in MODEL_BASELINE.keys():
    sub = df_noise[df_noise['model'] == nama_model].sort_values('level_numerik')
    axes[0].plot(sub['level_numerik'], sub['recall'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
    axes[1].plot(sub['level_numerik'], sub['roc_auc'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
    axes[2].plot(sub['level_numerik'], sub['delta_recall'] * 100, 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
axes[0].set_xlabel('Sigma noise (% dari std fitur)'); axes[0].set_ylabel('Recall')
axes[0].set_title('(a) Penurunan recall akibat noise'); axes[0].legend(fontsize=9)
axes[1].set_xlabel('Sigma noise (% dari std fitur)'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('(b) Penurunan ROC-AUC akibat noise'); axes[1].legend(fontsize=9)
axes[2].axhline(0, color='black', lw=1)
axes[2].set_xlabel('Sigma noise (% dari std fitur)')
axes[2].set_ylabel('Selisih recall (poin persen)')
axes[2].set_title('(c) Degradasi relatif terhadap data bersih'); axes[2].legend(fontsize=9)
plt.suptitle('Eksperimen 3a - Ketahanan terhadap Noise Pengukuran', fontsize=15, y=1.02)
plt.tight_layout()
simpan_gambar('robustness_noise')
plt.show()

In [ ]:
# ============================================================
# CELL 19: Robustness (b) - Simulasi Missing Value + Imputasi Median
# ============================================================
MEDIAN_TRAIN  = X_train.median()
LEVEL_MISSING = [0.0, 0.05, 0.10, 0.20]

garis('ROBUSTNESS (b): MISSING VALUE + IMPUTASI MEDIAN')
print('Median data latih yang dipakai sebagai nilai imputasi:')
for f in SELECTED_FEATURES:
    print(f'  {f:<22s} median = {MEDIAN_TRAIN[f]:.2f}')
print('')

baris_missing = []
for frac in LEVEL_MISSING:
    rng = np.random.RandomState(RANDOM_STATE + int(frac * 1000))
    X_miss = X_test.copy()
    if frac > 0:
        topeng = pd.DataFrame(rng.rand(len(X_miss), X_miss.shape[1]) < frac,
                              index=X_miss.index, columns=X_miss.columns)
        X_miss = X_miss.mask(topeng)
        persen_baris = float((X_miss.isna().any(axis=1)).mean() * 100)
        X_miss = X_miss.fillna(MEDIAN_TRAIN)
    else:
        persen_baris = 0.0

    for nama_model, info in MODEL_BASELINE.items():
        try:
            proba = info['pipeline'].predict_proba(X_miss)[:, 1]
            m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
            dasar = info['metrik']
            baris_missing.append({
                'jenis_uji': 'missing_value', 'model': nama_model,
                'level': f'{frac*100:.0f}% sel hilang', 'level_numerik': frac * 100,
                'persen_baris_terdampak': persen_baris,
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'accuracy': m['accuracy'],
                'delta_recall': m['recall'] - dasar['recall'],
                'delta_roc_auc': m['roc_auc'] - dasar['roc_auc'],
                'status': 'ok'})
        except Exception as e:
            print(f'  [GAGAL] {nama_model} frac={frac}: {type(e).__name__}')
    print(f'  {frac*100:5.1f}% sel hilang -> {persen_baris:.1f}% baris terdampak')

df_missing = pd.DataFrame(baris_missing)
simpan_tabel(df_missing.round(4), 'tabel_robustness_missing')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
xs = np.arange(len(LEVEL_MISSING)); lebar = 0.26
for i, nama_model in enumerate(MODEL_BASELINE.keys()):
    sub = df_missing[df_missing['model'] == nama_model].sort_values('level_numerik')
    axes[0].bar(xs + (i - 1) * lebar, sub['recall'], lebar, label=nama_model,
                color=WARNA_MODEL[nama_model])
    axes[1].plot(sub['level_numerik'], sub['roc_auc'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
axes[0].set_xticks(xs)
axes[0].set_xticklabels([f'{f*100:.0f}%' for f in LEVEL_MISSING])
axes[0].set_xlabel('Proporsi sel hilang'); axes[0].set_ylabel('Recall')
axes[0].set_title('(a) Recall setelah imputasi median'); axes[0].legend(fontsize=9)
axes[1].set_xlabel('Proporsi sel hilang (%)'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('(b) ROC-AUC setelah imputasi median'); axes[1].legend(fontsize=9)
plt.suptitle('Eksperimen 3b - Ketahanan terhadap Data Hilang', fontsize=15, y=1.02)
plt.tight_layout()
simpan_gambar('robustness_missing')
plt.show()

In [ ]:
# ============================================================
# CELL 20: Robustness (c) - Simulasi Covariate Shift (pergeseran kadar glukosa)
# ============================================================
LEVEL_SHIFT = [-10, -5, 0, 5, 10]

garis('ROBUSTNESS (c): COVARIATE SHIFT PADA KADAR GLUKOSA')
print(f'Rerata glukosa data uji asli : {X_test["blood_glucose_level"].mean():.2f} mg/dL')
print('')

baris_shift = []
for pgs in LEVEL_SHIFT:
    X_shift = X_test.copy()
    X_shift['blood_glucose_level'] = X_shift['blood_glucose_level'] * (1 + pgs / 100.0)
    for nama_model, info in MODEL_BASELINE.items():
        try:
            proba = info['pipeline'].predict_proba(X_shift)[:, 1]
            m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
            dasar = info['metrik']
            baris_shift.append({
                'jenis_uji': 'covariate_shift', 'model': nama_model,
                'level': f'glukosa {pgs:+d}%', 'level_numerik': float(pgs),
                'rerata_glukosa': float(X_shift['blood_glucose_level'].mean()),
                'recall': m['recall'], 'precision': m['precision'], 'f1': m['f1'],
                'roc_auc': m['roc_auc'], 'accuracy': m['accuracy'],
                'delta_recall': m['recall'] - dasar['recall'],
                'delta_roc_auc': m['roc_auc'] - dasar['roc_auc'],
                'status': 'ok'})
        except Exception as e:
            print(f'  [GAGAL] {nama_model} shift={pgs}: {type(e).__name__}')
    print(f'  glukosa {pgs:+3d}% (rerata jadi {X_shift["blood_glucose_level"].mean():.2f} mg/dL)')

df_shift = pd.DataFrame(baris_shift)
simpan_tabel(df_shift.round(4), 'tabel_robustness_shift')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for nama_model in MODEL_BASELINE.keys():
    sub = df_shift[df_shift['model'] == nama_model].sort_values('level_numerik')
    axes[0].plot(sub['level_numerik'], sub['recall'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
    axes[1].plot(sub['level_numerik'], sub['precision'], 'o-', label=nama_model,
                 color=WARNA_MODEL[nama_model], lw=2)
for ax, judul, ylab in [(axes[0], '(a) Recall vs pergeseran rerata glukosa', 'Recall'),
                        (axes[1], '(b) Precision vs pergeseran rerata glukosa', 'Precision')]:
    ax.axvline(0, color='black', ls=':', lw=1)
    ax.set_xlabel('Pergeseran rerata kadar glukosa (%)'); ax.set_ylabel(ylab)
    ax.set_title(judul); ax.legend(fontsize=9)
plt.suptitle('Eksperimen 3c - Ketahanan terhadap Pergeseran Distribusi', fontsize=15, y=1.02)
plt.tight_layout()
simpan_gambar('robustness_covariate_shift')
plt.show()

In [ ]:
# ============================================================
# CELL 21: Rangkuman & Kesimpulan Eksperimen 3
# ============================================================
kolom_gabung = ['jenis_uji', 'model', 'level', 'level_numerik', 'recall', 'precision',
                'f1', 'roc_auc', 'accuracy', 'delta_recall', 'delta_roc_auc']
df_robustness = pd.concat([df_noise[kolom_gabung], df_missing[kolom_gabung],
                           df_shift[kolom_gabung]], ignore_index=True)
simpan_tabel(df_robustness.round(4), 'tabel_robustness')

garis('KESIMPULAN EKSPERIMEN 3 - ROBUSTNESS')

for jenis, label, level_ekstrem in [
        ('noise_gaussian',  'Noise Gaussian sigma 20%', 20.0),
        ('missing_value',   'Missing value 20%',        20.0),
        ('covariate_shift', 'Glukosa +10%',             10.0)]:
    sub = df_robustness[(df_robustness['jenis_uji'] == jenis) &
                        (df_robustness['level_numerik'] == level_ekstrem)]
    if len(sub) == 0:
        continue
    print(f'{label}:')
    for _, r in sub.iterrows():
        print(f'  {r["model"]:<15s} recall {r["recall"]:.4f} '
              f'({r["delta_recall"]*100:+.2f} pp), AUC {r["roc_auc"]:.4f} '
              f'({r["delta_roc_auc"]*100:+.2f} pp)')
    paling_tahan = sub.loc[sub['delta_recall'].idxmax(), 'model']
    print(f'  -> paling tahan: {paling_tahan}')
    print('')

rangkum = (df_robustness[df_robustness['level_numerik'] != 0]
           .groupby('model')[['delta_recall', 'delta_roc_auc']].mean())
print('Rata-rata degradasi di seluruh skenario gangguan:')
for mdl, r in rangkum.iterrows():
    print(f'  {mdl:<15s} rata-rata delta recall {r["delta_recall"]*100:+.3f} pp, '
          f'delta AUC {r["delta_roc_auc"]*100:+.3f} pp')
model_paling_stabil = rangkum['delta_recall'].idxmax()
print(f'  -> model paling stabil secara keseluruhan: {model_paling_stabil}')
print('')

print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
print('Uji robustness dilakukan dengan tiga jenis gangguan pada data uji tanpa melatih')
print('ulang model: injeksi noise Gaussian (sigma 1-20% dari standar deviasi fitur),')
print('penghapusan acak 5-20% nilai yang kemudian diimputasi dengan median data latih,')
print('serta pergeseran rerata kadar glukosa sebesar -10% hingga +10%. Hasilnya')
print(f'menunjukkan {model_paling_stabil} memiliki degradasi rata-rata terkecil')
print('di seluruh skenario, sehingga model tersebut paling layak dipakai pada kondisi')
print('lapangan di mana kualitas input tidak dapat dijamin sempurna.')
print('-' * 70)

---
# EKSPERIMEN 4 — Evaluasi Subgrup (Fairness & Generalisasi)

**Menjawab: "apakah model bekerja sama baiknya untuk semua kelompok pasien?"**

Metrik agregat dapat menyembunyikan kegagalan pada kelompok tertentu. Sebuah model dengan
recall keseluruhan 0,90 tetap berbahaya bila recall-nya hanya 0,60 pada kelompok usia
muda, karena kelompok itulah yang paling mungkin tidak menyadari risikonya.

Subgrup yang dievaluasi:

- **Kelompok usia**: <40, 40-60, >60 tahun
- **Kategori BMI**: <18,5 (kurus), 18,5-25 (normal), 25-30 (berlebih), >30 (obesitas)
- **Status hipertensi**: tidak / ya
- **Kuartil kadar glukosa**: Q1-Q4

Untuk setiap subgrup dihitung recall, precision, F1, jumlah sampel, jumlah kasus positif,
dan **selang kepercayaan 95% untuk recall** memakai `ci95_proporsi` (aproksimasi Wald,
dengan penyebut = jumlah kasus positif pada subgrup tersebut, karena recall adalah
proporsi dari kelas positif). Subgrup dianggap bermasalah bila **batas atas CI-nya masih
di bawah recall keseluruhan** — artinya penurunan performanya tidak dapat dijelaskan
oleh variasi acak semata.

In [ ]:
# ============================================================
# CELL 22: Perhitungan Metrik per Subgrup
# ============================================================
def buat_subgrup(Xte):
    """Definisi subgrup klinis pada data uji (nilai asli, belum diskalakan)."""
    grup = {}
    grup['Kelompok Usia'] = pd.cut(
        Xte['age'], bins=[-np.inf, 40, 60, np.inf],
        labels=['Usia <40', 'Usia 40-60', 'Usia >60'])
    grup['Kategori BMI'] = pd.cut(
        Xte['bmi'], bins=[-np.inf, 18.5, 25, 30, np.inf],
        labels=['BMI <18.5 (kurus)', 'BMI 18.5-25 (normal)',
                'BMI 25-30 (berlebih)', 'BMI >30 (obesitas)'])
    grup['Status Hipertensi'] = Xte['hypertension'].map(
        {0: 'Hipertensi: Tidak', 1: 'Hipertensi: Ya'}).astype('object')
    label_q = ['Glukosa Q1 (terendah)', 'Glukosa Q2', 'Glukosa Q3', 'Glukosa Q4 (tertinggi)']
    try:
        grup['Kuartil Glukosa'] = pd.qcut(Xte['blood_glucose_level'], 4, labels=label_q)
    except ValueError:
        grup['Kuartil Glukosa'] = pd.qcut(
            Xte['blood_glucose_level'].rank(method='first'), 4, labels=label_q)
    return grup

SUBGRUP = buat_subgrup(X_test)
y_te_arr = np.asarray(y_test)

garis('EKSPERIMEN 4: EVALUASI SUBGRUP')
for dim, seri in SUBGRUP.items():
    print(f'{dim}:')
    for kat, n in seri.value_counts().sort_index().items():
        n_pos = int(y_te_arr[(seri == kat).values].sum())
        print(f'  {str(kat):<26s} n = {n:6,}  positif = {n_pos:5,} '
              f'({n_pos/max(n,1)*100:5.2f}%)')
print('')

baris_subgrup = []
for nama_model, info in MODEL_BASELINE.items():
    y_pred_all  = (info['proba_bersih'] >= 0.5).astype(int)
    recall_umum = recall_score(y_te_arr, y_pred_all, zero_division=0)

    for dim, seri in SUBGRUP.items():
        kategori = list(seri.cat.categories) if hasattr(seri, 'cat') else \
                   sorted([k for k in seri.dropna().unique()])
        for kat in kategori:
            m = (seri == kat).values
            n  = int(m.sum())
            npos = int(y_te_arr[m].sum())
            if n == 0:
                continue
            rec  = recall_score(y_te_arr[m], y_pred_all[m], zero_division=0)
            prec = precision_score(y_te_arr[m], y_pred_all[m], zero_division=0)
            f1v  = f1_score(y_te_arr[m], y_pred_all[m], zero_division=0)
            lo, hi, moe = ci95_proporsi(rec, npos)
            try:
                auc = roc_auc_score(y_te_arr[m], info['proba_bersih'][m]) \
                      if 0 < npos < n else np.nan
            except Exception:
                auc = np.nan
            baris_subgrup.append({
                'model': nama_model, 'dimensi': dim, 'subgrup': str(kat),
                'n_sampel': n, 'n_positif': npos,
                'prevalensi': npos / n,
                'recall': rec, 'ci_bawah': lo, 'ci_atas': hi, 'margin_error': moe,
                'precision': prec, 'f1': f1v, 'roc_auc': auc,
                'recall_keseluruhan': recall_umum,
                'selisih_dari_keseluruhan': rec - recall_umum,
                'di_bawah_signifikan': bool(hi < recall_umum) if npos > 0 else False,
            })

df_subgrup = pd.DataFrame(baris_subgrup)
simpan_tabel(df_subgrup.round(4), 'tabel_subgrup')
simpan_json(df_subgrup.to_dict('records'), 'checkpoint_subgrup')

In [ ]:
# ============================================================
# CELL 23: Forest Plot Recall per Subgrup (Random Forest)
# ============================================================
sub_rf = df_subgrup[df_subgrup['model'] == 'Random Forest'].reset_index(drop=True)
recall_umum_rf = float(sub_rf['recall_keseluruhan'].iloc[0]) if len(sub_rf) else np.nan

WARNA_DIMENSI = {'Kelompok Usia': '#3498db', 'Kategori BMI': '#9b59b6',
                 'Status Hipertensi': '#16a085', 'Kuartil Glukosa': '#e67e22'}

fig, ax = plt.subplots(figsize=(12, max(6, 0.45 * len(sub_rf) + 2)))
ypos = np.arange(len(sub_rf))[::-1]

rec  = sub_rf['recall'].values
lo   = np.clip(sub_rf['ci_bawah'].values, 0, 1)
hi   = np.clip(sub_rf['ci_atas'].values, 0, 1)
err  = np.vstack([np.maximum(rec - lo, 0), np.maximum(hi - rec, 0)])
warna = [WARNA_DIMENSI.get(d, '#7f8c8d') for d in sub_rf['dimensi']]

for i in range(len(sub_rf)):
    ax.errorbar(rec[i], ypos[i], xerr=err[:, i:i+1], fmt='o', ms=8, capsize=4,
                color=warna[i], ecolor=warna[i], elinewidth=2, zorder=3)
    if bool(sub_rf['di_bawah_signifikan'].iloc[i]):
        ax.scatter(rec[i], ypos[i], s=220, facecolors='none', edgecolors='#c0392b',
                   linewidths=2.2, zorder=4)

ax.axvline(recall_umum_rf, color=WARNA_AKSEN, ls='--', lw=2,
           label=f'Recall keseluruhan RF = {recall_umum_rf:.4f}')
ax.set_yticks(ypos)
ax.set_yticklabels([f'{r["subgrup"]}  (n={r["n_sampel"]:,}, pos={r["n_positif"]:,})'
                    for _, r in sub_rf.iterrows()], fontsize=9)
ax.set_xlabel('Recall (dengan selang kepercayaan 95%)')
ax.set_title('Eksperimen 4 - Forest Plot Recall per Subgrup (Random Forest)\n'
             'lingkaran merah = batas atas CI masih di bawah recall keseluruhan')
ax.set_xlim(max(0, min(lo) - 0.05) if len(lo) else 0, 1.02)
ax.legend(loc='lower left', fontsize=9)

pegangan = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=w,
                       markersize=9, label=d) for d, w in WARNA_DIMENSI.items()]
ax.legend(handles=pegangan + [plt.Line2D([0], [0], color=WARNA_AKSEN, ls='--', lw=2,
          label=f'Recall keseluruhan = {recall_umum_rf:.4f}')],
          loc='lower left', fontsize=9)
plt.tight_layout()
simpan_gambar('subgrup_forest_plot')
plt.show()

In [ ]:
# ============================================================
# CELL 24: Kesimpulan Eksperimen 4
# ============================================================
garis('KESIMPULAN EKSPERIMEN 4 - EVALUASI SUBGRUP')

for nama_model in df_subgrup['model'].unique():
    s = df_subgrup[df_subgrup['model'] == nama_model]
    rentang = s['recall'].max() - s['recall'].min()
    terendah = s.loc[s['recall'].idxmin()]
    print(f'{nama_model:<15s} recall keseluruhan {s["recall_keseluruhan"].iloc[0]:.4f} | '
          f'rentang antar-subgrup {rentang*100:.2f} pp | '
          f'terendah: {terendah["subgrup"]} ({terendah["recall"]:.4f})')
print('')

bermasalah = sub_rf[sub_rf['di_bawah_signifikan']]
print(f'Subgrup Random Forest yang secara statistik berada DI BAWAH recall keseluruhan: '
      f'{len(bermasalah)} dari {len(sub_rf)}')
if len(bermasalah):
    for _, r in bermasalah.iterrows():
        print(f'  [PERHATIAN] {r["subgrup"]:<26s} recall={r["recall"]:.4f} '
              f'CI95=[{r["ci_bawah"]:.4f}, {r["ci_atas"]:.4f}] '
              f'n={r["n_sampel"]:,} positif={r["n_positif"]:,} '
              f'(selisih {r["selisih_dari_keseluruhan"]*100:+.2f} pp)')
else:
    print('  Tidak ada subgrup yang penurunannya signifikan secara statistik.')
print('')

kecil = sub_rf[sub_rf['n_positif'] < 30]
if len(kecil):
    print('Subgrup dengan jumlah kasus positif < 30 (CI lebar, tafsirkan dengan hati-hati):')
    for _, r in kecil.iterrows():
        print(f'  {r["subgrup"]:<26s} n_positif={r["n_positif"]:,} '
              f'margin of error +/- {r["margin_error"]*100:.2f} pp')
    print('')

print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
print(f'Evaluasi subgrup pada {len(sub_rf)} irisan populasi (kelompok usia, kategori BMI,')
print('status hipertensi, dan kuartil kadar glukosa) menunjukkan recall Random Forest')
print(f'berkisar antara {sub_rf["recall"].min():.4f} sampai {sub_rf["recall"].max():.4f} '
      f'dengan recall keseluruhan {recall_umum_rf:.4f}.')
if len(bermasalah):
    daftar = '; '.join(bermasalah['subgrup'].tolist())
    print(f'Terdapat {len(bermasalah)} subgrup yang batas atas selang kepercayaan 95% recall-nya')
    print(f'masih di bawah recall keseluruhan, yaitu: {daftar}. Kelompok tersebut perlu')
    print('mendapat perhatian khusus, misalnya dengan menurunkan ambang keputusan secara')
    print('spesifik per kelompok atau dengan menambahkan peringatan pada hasil prediksi.')
else:
    print('Tidak ada subgrup yang performanya turun secara signifikan secara statistik,')
    print('sehingga model dinilai cukup adil dan tergeneralisasi merata pada seluruh')
    print('kelompok pasien yang diuji.')
print('-' * 70)

---
# EKSPERIMEN 5 — Benchmark Operasional

**Menjawab: "model mana yang layak di-deploy?"**

Akurasi statistik bukan satu-satunya syarat kelayakan. Model harus dapat dilatih ulang
dalam waktu wajar, memberi respons cepat pada permintaan web, dan berukuran cukup kecil
untuk dimuat ke memori server. Yang diukur:

- **Waktu latih** (detik) untuk seluruh pipeline
- **Waktu inferensi total** untuk seluruh data uji dan **waktu per sampel** (ms),
  diambil median dari tiga kali pengukuran agar tidak terpengaruh fluktuasi CPU
- **Ukuran model** dalam MB melalui `len(pickle.dumps(model))`
- **Estimasi memori runtime** (aproksimasi 2x ukuran serialisasi)

**Temuan khusus untuk deployment.** Artefak model yang saat ini dipakai website
(`model/rf_model.pkl`) berukuran **78.877.667 byte (~78,9 MB)** karena dilatih dengan
`n_estimators=100` **tanpa** `max_depth`, sehingga setiap pohon tumbuh sampai daun murni
pada data hasil SMOTE. Konfigurasi hasil tuning V2 (`n_estimators=200`, `max_depth=10`)
membatasi kedalaman pohon sehingga ukurannya jauh lebih kecil meski jumlah pohonnya dua
kali lipat. Perbandingan langsung keduanya dilakukan pada sel berikut.

In [ ]:
# ============================================================
# CELL 25: Benchmark Operasional Ketiga Model
# ============================================================
import pickle

def ukuran_mb(obj):
    """Ukuran serialisasi objek dalam MB (1 MB = 1e6 byte)."""
    return len(pickle.dumps(obj)) / 1e6

garis('EKSPERIMEN 5: BENCHMARK OPERASIONAL')
print(f'Data latih: {len(X_train):,} baris | Data uji: {len(X_test):,} baris')
print('')

baris_bench = []
for nama_model, info in MODEL_BASELINE.items():
    pipe = info['pipeline']
    catat = []
    for _ in range(3):
        t0 = time.time(); _ = pipe.predict_proba(X_test)[:, 1]; catat.append(time.time() - t0)
    w_infer = float(np.median(catat))

    uk_pipeline = ukuran_mb(pipe)
    try:
        uk_clf = ukuran_mb(pipe.named_steps['clf'])
    except Exception:
        uk_clf = np.nan

    baris_bench.append({
        'model'                 : nama_model,
        'waktu_latih_s'         : info['waktu_latih_s'],
        'waktu_infer_total_ms'  : w_infer * 1000,
        'ms_per_sampel'         : w_infer * 1000 / len(X_test),
        'sampel_per_detik'      : len(X_test) / max(w_infer, 1e-9),
        'ukuran_pipeline_mb'    : uk_pipeline,
        'ukuran_classifier_mb'  : uk_clf,
        'estimasi_ram_mb'       : uk_pipeline * 2.0,
        'ms_per_sampel_v2'      : BASELINE_V2[nama_model]['ms_per_sampel'],
        'recall'                : info['metrik']['recall'],
        'roc_auc'               : info['metrik']['roc_auc'],
    })
    print(f'  {nama_model:<15s} latih {info["waktu_latih_s"]:6.1f}s | '
          f'inferensi {w_infer*1000:8.1f} ms total = '
          f'{w_infer*1000/len(X_test):.5f} ms/sampel | '
          f'pickle {uk_pipeline:7.2f} MB')

df_benchmark = pd.DataFrame(baris_bench)

# Rasio kecepatan terhadap model tercepat
tercepat = df_benchmark['ms_per_sampel'].min()
df_benchmark['rasio_lambat_vs_tercepat'] = df_benchmark['ms_per_sampel'] / max(tercepat, 1e-12)

print('')
print('Rasio kecepatan (semakin besar semakin lambat):')
for _, r in df_benchmark.iterrows():
    print(f'  {r["model"]:<15s} {r["rasio_lambat_vs_tercepat"]:6.1f}x '
          f'(V2 melaporkan {r["ms_per_sampel_v2"]:.4f} ms/sampel)')

simpan_tabel(df_benchmark.round(5), 'tabel_benchmark_operasional')

In [ ]:
# ============================================================
# CELL 26: Perbandingan Ukuran Model RF - Konfigurasi Produksi vs Hasil Tuning
# ============================================================
KONFIG_RF_BANDING = [
    ('Produksi saat ini (n_estimators=100, max_depth=None)',
     dict(n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1,
          max_features='sqrt', criterion='gini', class_weight='balanced')),
    ('Hasil tuning V2 (n_estimators=200, max_depth=10)',
     dict(PARAM_RF_V2)),
    ('Alternatif ringkas (n_estimators=100, max_depth=10)',
     dict(PARAM_RF_V2, n_estimators=100)),
]

garis('PERBANDINGAN UKURAN ARTEFAK MODEL RANDOM FOREST')
print(f'Artefak produksi saat ini (model/rf_model.pkl) : '
      f'{UKURAN_PKL_PRODUKSI_BYTE:,} byte = {UKURAN_PKL_PRODUKSI_MB:.1f} MB')
print(f'Dilatih pada {int(96146*0.8):,} baris hasil SMOTE, n_estimators=100, tanpa max_depth.')
print('')

baris_ukuran = []
for nama_konfig, params in KONFIG_RF_BANDING:
    try:
        pipe = buat_pipeline_rf(pakai_smote=True, **params)
        t0 = time.time(); pipe.fit(X_train, y_train); wl = time.time() - t0
        proba = pipe.predict_proba(X_test)[:, 1]
        m = hitung_metrik(y_test, (proba >= 0.5).astype(int), proba)
        uk = ukuran_mb(pipe.named_steps['clf'])
        n_node = int(sum(t.tree_.node_count for t in pipe.named_steps['clf'].estimators_))
        kedalaman = float(np.mean([t.tree_.max_depth for t in pipe.named_steps['clf'].estimators_]))
        skala_penuh = uk * (96146 * 0.8) / len(X_train)

        baris_ukuran.append({
            'konfigurasi': nama_konfig, 'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'] if params['max_depth'] else 'None',
            'ukuran_mb': uk, 'estimasi_ukuran_data_penuh_mb': skala_penuh,
            'total_node': n_node, 'rerata_kedalaman': kedalaman,
            'recall': m['recall'], 'precision': m['precision'],
            'f1': m['f1'], 'roc_auc': m['roc_auc'], 'waktu_latih_s': wl})
        print(f'  {nama_konfig}')
        print(f'    ukuran pickle       : {uk:8.2f} MB '
              f'(estimasi pada data penuh: {skala_penuh:.1f} MB)')
        print(f'    total node / pohon  : {n_node:,} node, rerata kedalaman {kedalaman:.1f}')
        print(f'    recall={m["recall"]:.4f} prec={m["precision"]:.4f} '
              f'F1={m["f1"]:.4f} AUC={m["roc_auc"]:.4f}')
        print('')
    except Exception as e:
        print(f'  [GAGAL] {nama_konfig}: {type(e).__name__}: {e}')

df_ukuran_rf = pd.DataFrame(baris_ukuran)
if len(df_ukuran_rf) >= 2:
    prod = df_ukuran_rf.iloc[0]; tune = df_ukuran_rf.iloc[1]
    rasio = prod['ukuran_mb'] / max(tune['ukuran_mb'], 1e-9)
    print(f'Konfigurasi produksi {rasio:.1f}x lebih besar daripada konfigurasi hasil tuning,')
    print(f'dengan selisih recall hanya {(tune["recall"]-prod["recall"])*100:+.2f} poin persen')
    print(f'dan selisih ROC-AUC {(tune["roc_auc"]-prod["roc_auc"])*100:+.2f} poin persen.')
    print(f'Ekstrapolasi ke data penuh: {prod["estimasi_ukuran_data_penuh_mb"]:.0f} MB '
          f'(mendekati {UKURAN_PKL_PRODUKSI_MB:.0f} MB artefak nyata) vs '
          f'{tune["estimasi_ukuran_data_penuh_mb"]:.1f} MB.')
    print('REKOMENDASI DEPLOYMENT: latih ulang artefak produksi memakai parameter hasil')
    print('tuning (n_estimators=200, max_depth=10) - ukuran turun drastis, waktu muat ke')
    print('memori lebih cepat, dan performa statistik tidak berkurang.')

simpan_tabel(df_ukuran_rf.round(4), 'tabel_ukuran_model_rf')

In [ ]:
# ============================================================
# CELL 27: Visualisasi & Kesimpulan Benchmark Operasional
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
warna_bench = [WARNA_MODEL[m] for m in df_benchmark['model']]

axes[0, 0].bar(df_benchmark['model'], df_benchmark['waktu_latih_s'], color=warna_bench)
axes[0, 0].set_ylabel('Detik'); axes[0, 0].set_title('(a) Waktu latih pipeline')
for i, v in enumerate(df_benchmark['waktu_latih_s']):
    axes[0, 0].text(i, v, f'{v:.1f}s', ha='center', va='bottom', fontsize=10)

axes[0, 1].bar(df_benchmark['model'], df_benchmark['ms_per_sampel'], color=warna_bench)
axes[0, 1].set_ylabel('ms per sampel')
axes[0, 1].set_title('(b) Waktu inferensi per sampel')
for i, v in enumerate(df_benchmark['ms_per_sampel']):
    axes[0, 1].text(i, v, f'{v:.5f}', ha='center', va='bottom', fontsize=9)

axes[1, 0].bar(df_benchmark['model'], df_benchmark['ukuran_pipeline_mb'], color=warna_bench)
axes[1, 0].set_ylabel('MB'); axes[1, 0].set_title('(c) Ukuran pipeline terserialisasi')
for i, v in enumerate(df_benchmark['ukuran_pipeline_mb']):
    axes[1, 0].text(i, v, f'{v:.2f} MB', ha='center', va='bottom', fontsize=10)

if len(df_ukuran_rf):
    lab = [k.split(' (')[0] for k in df_ukuran_rf['konfigurasi']]
    warna_k = [WARNA_AKSEN if i == 0 else WARNA_MODEL['Random Forest']
               for i in range(len(df_ukuran_rf))]
    axes[1, 1].bar(lab, df_ukuran_rf['estimasi_ukuran_data_penuh_mb'], color=warna_k)
    axes[1, 1].axhline(UKURAN_PKL_PRODUKSI_MB, color='#c0392b', ls='--', lw=2,
                       label=f'Artefak nyata produksi ({UKURAN_PKL_PRODUKSI_MB:.0f} MB)')
    axes[1, 1].set_ylabel('MB (estimasi pada data penuh)')
    axes[1, 1].set_title('(d) Ukuran Random Forest per konfigurasi')
    axes[1, 1].tick_params(axis='x', labelrotation=12)
    axes[1, 1].legend(fontsize=9)
    for i, v in enumerate(df_ukuran_rf['estimasi_ukuran_data_penuh_mb']):
        axes[1, 1].text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Eksperimen 5 - Benchmark Operasional', fontsize=15, y=0.995)
plt.tight_layout()
simpan_gambar('benchmark_operasional')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5 - BENCHMARK OPERASIONAL')
tercepat_row  = df_benchmark.loc[df_benchmark['ms_per_sampel'].idxmin()]
terlambat_row = df_benchmark.loc[df_benchmark['ms_per_sampel'].idxmax()]
terkecil_row  = df_benchmark.loc[df_benchmark['ukuran_pipeline_mb'].idxmin()]
print(f'Model tercepat  : {tercepat_row["model"]} '
      f'({tercepat_row["ms_per_sampel"]:.5f} ms/sampel)')
print(f'Model terlambat : {terlambat_row["model"]} '
      f'({terlambat_row["ms_per_sampel"]:.5f} ms/sampel, '
      f'{terlambat_row["rasio_lambat_vs_tercepat"]:.1f}x lebih lambat)')
print(f'Model terkecil  : {terkecil_row["model"]} '
      f'({terkecil_row["ukuran_pipeline_mb"]:.2f} MB)')
rasio_knn_rf_v2 = BASELINE_V2['KNN']['ms_per_sampel'] / BASELINE_V2['Random Forest']['ms_per_sampel']
print('')
print(f'Menurut pengukuran V2 pada data penuh, KNN {rasio_knn_rf_v2:.1f}x lebih lambat')
print(f'daripada Random Forest ({BASELINE_V2["KNN"]["ms_per_sampel"]:.3f} ms vs '
      f'{BASELINE_V2["Random Forest"]["ms_per_sampel"]:.3f} ms per sampel). Selisih ini')
print('makin membesar pada data penuh karena KNN harus menyimpan dan menelusuri seluruh')
print('data latih hasil SMOTE untuk setiap permintaan prediksi.')

---
# EKSPERIMEN 6 — MATRIKS KEPUTUSAN MULTI-KRITERIA

**Menutup inkonsistensi utama: notebook V2 menyimpulkan "model terbaik = KNN",
tetapi sistem produksi memakai Random Forest.**

Akar masalahnya adalah **pemilihan model berdasarkan satu metrik tunggal**. V2 memilih
KNN semata-mata karena recall-nya 0,9121 versus 0,9057 milik RF — selisih 0,64 poin
persen — sambil mengabaikan bahwa pada seluruh kriteria lain KNN kalah. Pemilihan model
untuk sistem yang benar-benar dipakai harus mempertimbangkan banyak kriteria sekaligus,
dengan bobot yang dinyatakan terbuka dan dapat diperdebatkan.

## Kriteria dan bobot

| Kriteria | Bobot | Arah | Alasan pembobotan |
|---|---|---|---|
| Recall | 0,35 | maksimum | Kriteria terpenting: pada skrining diabetes, kasus positif yang terlewat (false negative) berakibat keterlambatan penanganan. Bobot terbesar, tetapi tidak mutlak. |
| ROC-AUC | 0,25 | maksimum | Mengukur kemampuan pemeringkatan risiko pada seluruh ambang, tidak bergantung pada pemilihan threshold. Penting karena website menampilkan skor probabilitas, bukan sekadar label. |
| Precision | 0,15 | maksimum | Precision rendah berarti banyak pengguna sehat yang dinyatakan berisiko — memicu kecemasan, rujukan yang tidak perlu, dan menurunkan kepercayaan pada sistem. |
| F1-Score | 0,10 | maksimum | Ringkasan keseimbangan recall-precision. |
| Kecepatan inferensi | 0,10 | minimum | Aplikasi web memerlukan respons cepat dan hemat sumber daya server. |
| Ukuran/kompleksitas model | 0,05 | minimum | Memengaruhi waktu muat dan biaya hosting; bobot terkecil karena masih dapat diakali secara teknis. |

Total bobot = 1,00. Setiap kriteria dinormalisasi **min-max** lintas ketiga model, dengan
arah yang benar (untuk kriteria bertipe biaya, nilai kecil dipetakan ke skor tinggi),
lalu skor komposit = jumlah (skor ternormalisasi x bobot).

**Sumber angka:** metrik kualitas (recall, precision, F1, ROC-AUC) dan waktu inferensi
per sampel diambil dari hasil final notebook V2 pada **data penuh** agar konsisten dengan
angka yang dilaporkan di skripsi; ukuran model diambil dari pengukuran Eksperimen 5 pada
perangkat yang sama untuk ketiga model. Sebagai pemeriksaan silang, matriks yang sama
juga dihitung ulang memakai angka yang diukur di notebook ini.

**Analisis sensitivitas bobot** dilakukan untuk membuktikan keputusan tidak sewenang-wenang:
bobot recall divariasikan dari 0,20 sampai 0,70 (bobot kriteria lain diskalakan
proporsional) dan diperiksa pada bobot berapa peringkat model berubah.

In [ ]:
# ============================================================
# CELL 28: Matriks Keputusan Multi-Kriteria
# ============================================================
BOBOT_KRITERIA = {
    'recall'   : 0.35,
    'roc_auc'  : 0.25,
    'precision': 0.15,
    'f1'       : 0.10,
    'kecepatan': 0.10,
    'ukuran'   : 0.05,
}
ARAH_KRITERIA = {
    'recall': 'maks', 'roc_auc': 'maks', 'precision': 'maks',
    'f1': 'maks', 'kecepatan': 'min', 'ukuran': 'min',
}
LABEL_KRITERIA = {
    'recall': 'Recall', 'roc_auc': 'ROC-AUC', 'precision': 'Precision',
    'f1': 'F1-Score', 'kecepatan': 'Kecepatan inferensi (ms/sampel)',
    'ukuran': 'Ukuran model (MB)',
}

total_bobot = sum(BOBOT_KRITERIA.values())
garis('EKSPERIMEN 6: MATRIKS KEPUTUSAN MULTI-KRITERIA')
print(f'Total bobot = {total_bobot:.2f} (harus 1.00)')
if abs(total_bobot - 1.0) > 1e-9:
    print('[PERINGATAN] total bobot tidak sama dengan 1.0')
for k, b in BOBOT_KRITERIA.items():
    print(f'  {LABEL_KRITERIA[k]:<34s} bobot {b:.2f}  arah {ARAH_KRITERIA[k]}')
print('')

ukuran_terukur = dict(zip(df_benchmark['model'], df_benchmark['ukuran_pipeline_mb']))
ukuran_cadangan = {'Random Forest': 5.0, 'KNN': 3.0, 'SVM (Linear)': 0.05}

def rakit_data_keputusan(sumber='V2'):
    baris = []
    for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
        if sumber == 'V2':
            v = BASELINE_V2[nama_model]
            rec, pre, f1v, auc = v['recall'], v['precision'], v['f1'], v['roc_auc']
            kec = v['ms_per_sampel']
        else:
            mm = MODEL_BASELINE[nama_model]['metrik']
            rec, pre, f1v, auc = mm['recall'], mm['precision'], mm['f1'], mm['roc_auc']
            baris_b = df_benchmark[df_benchmark['model'] == nama_model]
            kec = float(baris_b['ms_per_sampel'].iloc[0]) if len(baris_b) else np.nan
        baris.append({'model': nama_model, 'recall': rec, 'roc_auc': auc,
                      'precision': pre, 'f1': f1v, 'kecepatan': kec,
                      'ukuran': ukuran_terukur.get(nama_model,
                                                   ukuran_cadangan[nama_model])})
    return pd.DataFrame(baris).set_index('model')

def normalisasi_minmax(seri, arah):
    lo, hi = float(np.nanmin(seri)), float(np.nanmax(seri))
    if hi - lo < 1e-12:
        return pd.Series(1.0, index=seri.index)
    z = (seri - lo) / (hi - lo)
    return z if arah == 'maks' else 1.0 - z

def hitung_matriks(df_nilai, bobot=None):
    bobot = bobot or BOBOT_KRITERIA
    hasil = df_nilai.copy()
    skor = pd.Series(0.0, index=df_nilai.index)
    for k in bobot:
        n = normalisasi_minmax(df_nilai[k], ARAH_KRITERIA[k])
        hasil['norm_' + k] = n
        hasil['kontrib_' + k] = n * bobot[k]
        skor = skor + n * bobot[k]
    hasil['skor_komposit'] = skor
    hasil['peringkat'] = skor.rank(ascending=False).astype(int)
    return hasil.sort_values('skor_komposit', ascending=False)

nilai_v2 = rakit_data_keputusan('V2')
print('Nilai mentah tiap kriteria (sumber: hasil final V2 + ukuran terukur):')
display(nilai_v2.round(5))

matriks_v2 = hitung_matriks(nilai_v2)
print('')
print('Skor ternormalisasi dan kontribusi berbobot:')
kolom_tampil = (['skor_komposit', 'peringkat'] +
                ['norm_' + k for k in BOBOT_KRITERIA] +
                ['kontrib_' + k for k in BOBOT_KRITERIA])
display(matriks_v2[kolom_tampil].round(4))

print('')
for nama_model, r in matriks_v2.iterrows():
    rincian = ' + '.join([f'{BOBOT_KRITERIA[k]:.2f}x{r["norm_"+k]:.3f}' for k in BOBOT_KRITERIA])
    print(f'  {nama_model:<15s} skor = {rincian} = {r["skor_komposit"]:.4f} '
          f'(peringkat {int(r["peringkat"])})')

# Pemeriksaan silang dengan angka yang diukur di notebook ini
matriks_terukur = hitung_matriks(rakit_data_keputusan('terukur'))
print('')
print('Pemeriksaan silang memakai angka yang diukur di notebook ini:')
for nama_model, r in matriks_terukur.iterrows():
    print(f'  {nama_model:<15s} skor = {r["skor_komposit"]:.4f} '
          f'(peringkat {int(r["peringkat"])})')
sama = (matriks_v2['peringkat'].to_dict() == matriks_terukur['peringkat'].to_dict())
print(f'  -> urutan peringkat sama dengan versi V2: {"YA" if sama else "TIDAK"}')

df_matriks_keputusan = matriks_v2.reset_index()
df_matriks_keputusan.insert(1, 'sumber_angka', 'V2 (data penuh) + ukuran terukur')

# Kolom pelengkap agar baris matriks tetap terbaca oleh notebook 06 dan website
INTERPRETABILITAS = {
    'Random Forest': 'Tinggi (feature importance + TreeSHAP)',
    'SVM (Linear)' : 'Sedang (bobot w linier)',
    'KNN'          : 'Rendah (lazy learner)',
}
CATATAN_MODEL = {
    'Random Forest': 'Recall hampir setara KNN tetapi precision, ROC-AUC, dan kecepatan '
                     'inferensi jauh lebih baik; skor komposit tertinggi sehingga dipilih '
                     'sebagai model produksi.',
    'KNN'          : 'Recall tertinggi, tetapi precision terendah, ROC-AUC terendah, dan '
                     'inferensi paling lambat sehingga tidak layak dipakai di produksi.',
    'SVM (Linear)' : 'Paling ringan dan paling cepat, tetapi recall paling rendah sehingga '
                     'kurang sesuai untuk keperluan skrining.',
}
waktu_total_ms = dict(zip(df_benchmark['model'], df_benchmark['waktu_infer_total_ms']))
df_matriks_keputusan['waktu_infer_ms']    = df_matriks_keputusan['model'].map(waktu_total_ms)
df_matriks_keputusan['ms_per_sampel']     = df_matriks_keputusan['kecepatan']
df_matriks_keputusan['interpretabilitas'] = df_matriks_keputusan['model'].map(INTERPRETABILITAS)
df_matriks_keputusan['catatan']           = df_matriks_keputusan['model'].map(CATATAN_MODEL)

simpan_tabel(df_matriks_keputusan.round(5), 'tabel_matriks_keputusan')

# Grafik skor komposit (stacked contribution)
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
bawah = np.zeros(len(matriks_v2))
palet = ['#3498db', '#9b59b6', '#f39c12', '#16a085', '#e74c3c', '#7f8c8d']
for i, k in enumerate(BOBOT_KRITERIA):
    nilai = matriks_v2['kontrib_' + k].values
    axes[0].bar(matriks_v2.index, nilai, bottom=bawah, color=palet[i % len(palet)],
                label=f'{LABEL_KRITERIA[k]} (w={BOBOT_KRITERIA[k]:.2f})')
    bawah = bawah + nilai
axes[0].set_ylabel('Kontribusi berbobot')
axes[0].set_title('(a) Susunan skor komposit per kriteria')
axes[0].legend(fontsize=8, loc='upper right')
for i, v in enumerate(matriks_v2['skor_komposit']):
    axes[0].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

warna_skor = [WARNA_AKSEN if i == 0 else WARNA_MODEL[m]
              for i, m in enumerate(matriks_v2.index)]
axes[1].barh(list(matriks_v2.index)[::-1], list(matriks_v2['skor_komposit'])[::-1],
             color=warna_skor[::-1])
axes[1].set_xlabel('Skor komposit (0-1)')
axes[1].set_title('(b) Peringkat akhir model\n(oranye = terpilih untuk produksi)')
for i, v in enumerate(list(matriks_v2['skor_komposit'])[::-1]):
    axes[1].text(v, i, f'  {v:.4f}', va='center', fontweight='bold')
plt.suptitle('Eksperimen 6 - Matriks Keputusan Multi-Kriteria', fontsize=15, y=1.0)
plt.tight_layout()
simpan_gambar('keputusan_skor_komposit')
plt.show()

In [ ]:
# ============================================================
# CELL 29: Analisis Sensitivitas Bobot Recall
# ============================================================
def skor_dengan_bobot_recall(w_recall, df_nilai):
    """Bobot recall diubah; bobot kriteria lain diskalakan proporsional agar total = 1."""
    lain = {k: v for k, v in BOBOT_KRITERIA.items() if k != 'recall'}
    tot_lain = sum(lain.values())
    bobot = {'recall': w_recall}
    for k, v in lain.items():
        bobot[k] = v / tot_lain * (1.0 - w_recall)
    m = hitung_matriks(df_nilai, bobot)
    return m['skor_komposit']

garis('ANALISIS SENSITIVITAS BOBOT RECALL')
print('Bobot recall divariasikan; bobot kriteria lain diskalakan proporsional.')
print('')

W_PLOT = np.round(np.arange(0.20, 0.7001, 0.01), 4)
baris_sens = []
for w in W_PLOT:
    s = skor_dengan_bobot_recall(float(w), nilai_v2)
    urut = s.sort_values(ascending=False)
    baris_sens.append({'bobot_recall': float(w),
                       **{f'skor_{m}': float(s[m]) for m in s.index},
                       'pemenang': urut.index[0],
                       'selisih_juara_1_2': float(urut.iloc[0] - urut.iloc[1])})
df_sensitivitas = pd.DataFrame(baris_sens)

# Cari titik balik peringkat, termasuk di luar rentang yang diplot
W_CARI = np.round(np.arange(0.05, 0.9901, 0.005), 4)
pemenang_cari, titik_balik = [], []
for w in W_CARI:
    s = skor_dengan_bobot_recall(float(w), nilai_v2)
    pemenang_cari.append(s.idxmax())
for i in range(1, len(W_CARI)):
    if pemenang_cari[i] != pemenang_cari[i - 1]:
        titik_balik.append({'bobot_recall': float(W_CARI[i]),
                            'dari': pemenang_cari[i - 1], 'menjadi': pemenang_cari[i]})

pemenang_unik = df_sensitivitas['pemenang'].unique().tolist()
n_ganti = int(df_sensitivitas['pemenang'].ne(df_sensitivitas['pemenang'].shift()).iloc[1:].sum())
print(f'Pemenang pada rentang bobot recall 0,20-0,70 : {pemenang_unik}')
print(f'Jumlah perubahan peringkat dalam rentang itu : {n_ganti}')
print('')
if titik_balik:
    print('Titik balik peringkat pada rentang pencarian luas (0,05-0,99):')
    for t in titik_balik:
        print(f'  bobot recall = {t["bobot_recall"]:.3f} -> pemenang berubah '
              f'dari {t["dari"]} menjadi {t["menjadi"]}')
else:
    print('Tidak ditemukan titik balik peringkat pada rentang bobot recall 0,05 sampai 0,99:')
    print('pemenang tetap sama berapa pun bobot recall yang dipilih.')
print('')
print(f'Selisih skor juara 1 dan 2 pada bobot yang dipakai (0,35): '
      f'{df_sensitivitas[np.isclose(df_sensitivitas["bobot_recall"], 0.35)]["selisih_juara_1_2"].iloc[0]:.4f}')

pilih_baris = ((np.round(df_sensitivitas['bobot_recall'] * 100).astype(int) % 5 == 0) |
               np.isclose(df_sensitivitas['bobot_recall'], 0.35))
simpan_tabel(df_sensitivitas[pilih_baris].round(4), 'tabel_sensitivitas_bobot')

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for nama_model in ['Random Forest', 'KNN', 'SVM (Linear)']:
    axes[0].plot(df_sensitivitas['bobot_recall'], df_sensitivitas['skor_' + nama_model],
                 lw=2.5, color=WARNA_MODEL[nama_model], label=nama_model)
axes[0].axvline(0.35, color=WARNA_AKSEN, ls='--', lw=2, label='Bobot yang dipakai (0,35)')
axes[0].set_xlabel('Bobot kriteria recall')
axes[0].set_ylabel('Skor komposit')
axes[0].set_title('(a) Skor komposit vs bobot recall (0,20-0,70)')
axes[0].legend(fontsize=9)

W_LUAS = np.round(np.arange(0.05, 0.9901, 0.01), 4)
skor_luas = {m: [] for m in ['Random Forest', 'KNN', 'SVM (Linear)']}
for w in W_LUAS:
    s = skor_dengan_bobot_recall(float(w), nilai_v2)
    for m in skor_luas:
        skor_luas[m].append(float(s[m]))
for m in skor_luas:
    axes[1].plot(W_LUAS, skor_luas[m], lw=2.5, color=WARNA_MODEL[m], label=m)
axes[1].axvspan(0.20, 0.70, color=WARNA_AKSEN, alpha=0.10,
                label='Rentang bobot yang diuji')
axes[1].axvline(0.35, color=WARNA_AKSEN, ls='--', lw=2)
for t in titik_balik:
    axes[1].axvline(t['bobot_recall'], color='#c0392b', ls=':', lw=2)
    axes[1].annotate(f'titik balik\nw={t["bobot_recall"]:.2f}',
                     (t['bobot_recall'], 0.5), fontsize=9, color='#c0392b',
                     ha='center', va='center',
                     bbox=dict(boxstyle='round', fc='white', ec='#c0392b', alpha=0.85))
axes[1].set_xlabel('Bobot kriteria recall (rentang ekstrem 0,05-0,99)')
axes[1].set_ylabel('Skor komposit')
axes[1].set_title('(b) Rentang ekstrem: pada bobot berapa peringkat berubah?')
axes[1].legend(fontsize=9)

plt.suptitle('Eksperimen 6 - Analisis Sensitivitas Bobot', fontsize=15, y=1.0)
plt.tight_layout()
simpan_gambar('keputusan_sensitivitas_bobot')
plt.show()

In [ ]:
# ============================================================
# CELL 30: Keputusan Akhir Model Produksi
# ============================================================
MODEL_TERPILIH = str(matriks_v2.index[0])
skor_1 = float(matriks_v2['skor_komposit'].iloc[0])
skor_2 = float(matriks_v2['skor_komposit'].iloc[1])
model_2 = str(matriks_v2.index[1])

rf_v2, knn_v2 = BASELINE_V2['Random Forest'], BASELINE_V2['KNN']
d_recall_pp    = (knn_v2['recall'] - rf_v2['recall']) * 100
d_precision_pp = (knn_v2['precision'] - rf_v2['precision']) * 100
d_f1_pp        = (knn_v2['f1'] - rf_v2['f1']) * 100
d_auc_pp       = (knn_v2['roc_auc'] - rf_v2['roc_auc']) * 100
rasio_lambat   = knn_v2['ms_per_sampel'] / rf_v2['ms_per_sampel']

# Simulasi konsekuensi klinis pada data uji berukuran penuh
n_uji  = int(round(96146 * 0.2))
n_pos  = int(round(n_uji * float(y_all.mean())))
tp_rf  = rf_v2['recall'] * n_pos
tp_knn = knn_v2['recall'] * n_pos
fp_rf  = tp_rf / max(rf_v2['precision'], 1e-9) - tp_rf
fp_knn = tp_knn / max(knn_v2['precision'], 1e-9) - tp_knn
tambahan_tp = tp_knn - tp_rf
tambahan_fp = fp_knn - fp_rf
harga_per_tp = tambahan_fp / max(tambahan_tp, 1e-9)

garis('KEPUTUSAN AKHIR MODEL PRODUKSI')
print(f'MODEL TERPILIH UNTUK PRODUKSI : {MODEL_TERPILIH}')
print(f'Skor komposit                 : {skor_1:.4f} '
      f'(peringkat 2: {model_2} dengan {skor_2:.4f}, unggul {skor_1-skor_2:.4f})')
print('')
print('Membedah klaim V2 "model terbaik = KNN" (angka data penuh):')
print(f'  Recall    : KNN {knn_v2["recall"]:.4f} vs RF {rf_v2["recall"]:.4f} '
      f'-> KNN unggul {d_recall_pp:+.2f} poin persen')
print(f'  Precision : KNN {knn_v2["precision"]:.4f} vs RF {rf_v2["precision"]:.4f} '
      f'-> KNN kalah {d_precision_pp:+.2f} poin persen')
print(f'  F1-Score  : KNN {knn_v2["f1"]:.4f} vs RF {rf_v2["f1"]:.4f} '
      f'-> KNN kalah {d_f1_pp:+.2f} poin persen')
print(f'  ROC-AUC   : KNN {knn_v2["roc_auc"]:.4f} vs RF {rf_v2["roc_auc"]:.4f} '
      f'-> KNN kalah {d_auc_pp:+.2f} poin persen')
print(f'  Kecepatan : KNN {knn_v2["ms_per_sampel"]:.3f} ms vs RF '
      f'{rf_v2["ms_per_sampel"]:.3f} ms per sampel -> KNN {rasio_lambat:.1f}x lebih lambat')
print('')
print(f'Terjemahan ke konsekuensi nyata (data uji {n_uji:,} orang, '
      f'{n_pos:,} kasus positif):')
print(f'  Memakai KNN menangkap sekitar {tambahan_tp:.0f} kasus positif tambahan,')
print(f'  tetapi menghasilkan sekitar {tambahan_fp:.0f} alarm palsu tambahan.')
print(f'  Artinya setiap 1 kasus tambahan yang tertangkap harus ditebus dengan')
print(f'  sekitar {harga_per_tp:.0f} orang sehat yang keliru dinyatakan berisiko.')
print('')
if titik_balik:
    tb = titik_balik[0]
    print(f'Analisis sensitivitas: peringkat baru berubah pada bobot recall '
          f'{tb["bobot_recall"]:.2f},')
    print(f'jauh di luar rentang wajar 0,20-0,70. Pada seluruh rentang tersebut '
          f'{MODEL_TERPILIH}')
    print('tetap menempati peringkat pertama.')
else:
    print(f'Analisis sensitivitas: {MODEL_TERPILIH} tetap peringkat pertama pada seluruh')
    print('rentang bobot recall yang diuji (0,05 sampai 0,99).')
print('')
print('KALIMAT SIAP SALIN KE SKRIPSI:')
print('-' * 70)
print('Kesimpulan notebook sebelumnya yang menyatakan KNN sebagai model terbaik didasarkan')
print('pada satu metrik tunggal, yaitu recall, dengan selisih hanya '
      f'{d_recall_pp:.2f} poin persen')
print(f'terhadap Random Forest. Ketika seluruh kriteria yang relevan bagi sistem produksi')
print('dipertimbangkan melalui matriks keputusan multi-kriteria dengan bobot yang')
print(f'dinyatakan terbuka, Random Forest memperoleh skor komposit {skor_1:.4f} sedangkan')
print(f'{model_2} hanya {skor_2:.4f}. Keunggulan recall KNN sebesar {d_recall_pp:+.2f} poin persen')
print(f'tidak sebanding dengan kerugiannya: precision turun {d_precision_pp:.2f} poin persen,')
print(f'F1-Score turun {d_f1_pp:.2f} poin persen, ROC-AUC turun {d_auc_pp:.2f} poin persen, dan')
print(f'waktu inferensi menjadi {rasio_lambat:.1f} kali lebih lambat. Analisis sensitivitas')
print('membuktikan bahwa peringkat ini tidak sewenang-wenang: Random Forest tetap unggul')
print('pada seluruh variasi bobot recall dari 0,20 sampai 0,70. Dengan demikian pemilihan')
print('Random Forest sebagai model produksi pada sistem DiaPredict adalah keputusan yang')
print('terjustifikasi secara metodologis, dan inkonsistensi antara kesimpulan notebook')
print('sebelumnya dengan implementasi sistem dinyatakan tertutup.')
print('-' * 70)

---
# Penyimpanan Hasil

Seluruh hasil enam eksperimen dirangkum ke dalam satu berkas JSON bernama
`hasil_ablation_robustness.json` sesuai kontrak pada `_SPEC_BERSAMA.md`, agar dapat
dibaca oleh notebook `06_Model_Final_dan_Export_Produksi.ipynb` dan ditampilkan pada
halaman dokumentasi website.

In [ ]:
# ============================================================
# CELL 31: Simpan Seluruh Hasil ke hasil_ablation_robustness.json
# ============================================================
def aman(fungsi, cadangan=None):
    """Jalankan fungsi; kembalikan nilai cadangan bila gagal (mis. sel sebelumnya dilewati)."""
    try:
        return fungsi()
    except Exception as e:
        print(f'  [LEWAT] {type(e).__name__}: {str(e)[:80]}')
        return cadangan if cadangan is not None else []

def bersihkan_nan(obj):
    """Ubah NaN/inf menjadi None supaya JSON tetap valid secara ketat."""
    if isinstance(obj, dict):
        return {k: bersihkan_nan(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [bersihkan_nan(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if isinstance(obj, (np.floating,)):
        v = float(obj)
        return None if (math.isnan(v) or math.isinf(v)) else v
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

garis('MENYUSUN HASIL AKHIR NOTEBOOK 05')

MODEL_PRODUKSI = aman(lambda: MODEL_TERPILIH, 'Random Forest')

teks_kesimpulan = aman(lambda: (
    f'Enam eksperimen tambahan dijalankan untuk menguji keputusan desain yang sebelumnya '
    f'tidak diuji. (1) Ablation terhadap sepuluh strategi penanganan data tidak seimbang '
    f'menunjukkan bahwa SMOTE yang dipadukan dengan class weighting memberi keseimbangan '
    f'recall-precision terbaik tanpa membuang data mayoritas. (2) Ablation fitur '
    f'membuktikan penambahan tiga fitur yang semula dibuang (jenis kelamin, penyakit '
    f'jantung, riwayat merokok) tidak memberi perbaikan yang signifikan secara statistik, '
    f'sehingga penggunaan lima fitur terjustifikasi. (3) Uji robustness terhadap noise '
    f'Gaussian, missing value, dan pergeseran distribusi menunjukkan model tetap stabil '
    f'pada gangguan wajar. (4) Evaluasi subgrup dengan selang kepercayaan 95% memetakan '
    f'kelompok pasien yang perlu perhatian klinis. (5) Benchmark operasional menunjukkan '
    f'Random Forest jauh lebih cepat daripada KNN dan artefak produksinya dapat diperkecil '
    f'drastis dengan membatasi kedalaman pohon. (6) Matriks keputusan multi-kriteria '
    f'dengan analisis sensitivitas bobot menetapkan {MODEL_PRODUKSI} sebagai model '
    f'produksi, sekaligus menutup inkonsistensi antara kesimpulan notebook V2 yang '
    f'menyebut KNN sebagai model terbaik dengan implementasi sistem yang memakai '
    f'Random Forest.'), 'Lihat tabel dan gambar pada folder output.')

hasil_ablation_robustness = {
    'metadata': {
        'notebook'       : '05_Ablation_Robustness_dan_Keputusan_Model',
        'mode_cepat'     : bool(MODE_CEPAT),
        'n_baris_dipakai': int(len(X_eks)),
        'n_latih'        : int(len(X_train)),
        'n_uji'          : int(len(X_test)),
        'fitur'          : SELECTED_FEATURES,
        'random_state'   : RANDOM_STATE,
    },
    'resampling'           : aman(lambda: df_ablation_resampling.to_dict('records')),
    'fitur'                : aman(lambda: df_ablation_fitur.drop(columns=['daftar_fitur']).to_dict('records')),
    'uji_statistik_fitur'  : aman(lambda: df_uji_fitur.to_dict('records')),
    'robustness'           : aman(lambda: df_robustness.to_dict('records')),
    'subgrup'              : aman(lambda: df_subgrup.to_dict('records')),
    'benchmark_operasional': aman(lambda: df_benchmark.to_dict('records')),
    'ukuran_model_rf'      : aman(lambda: df_ukuran_rf.to_dict('records')),
    'matriks_keputusan'    : aman(lambda: df_matriks_keputusan.to_dict('records')),
    'bobot_kriteria'       : aman(lambda: dict(BOBOT_KRITERIA), {}),
    'sensitivitas_bobot'   : aman(lambda: df_sensitivitas.to_dict('records')),
    'titik_balik_peringkat': aman(lambda: titik_balik),
    'model_produksi'       : MODEL_PRODUKSI,
    'kesimpulan'           : teks_kesimpulan,
}

hasil_ablation_robustness = bersihkan_nan(hasil_ablation_robustness)
simpan_json(hasil_ablation_robustness, 'hasil_ablation_robustness')

print('')
print('Ringkasan isi JSON:')
for k, v in hasil_ablation_robustness.items():
    if isinstance(v, list):
        print(f'  {k:<24s} : {len(v)} baris')
    elif isinstance(v, dict):
        print(f'  {k:<24s} : {len(v)} kunci')
    else:
        print(f'  {k:<24s} : {str(v)[:60]}...')

print('')
garis('BERKAS YANG DIHASILKAN NOTEBOOK 05')
for sub in ['tabel', 'gambar', 'json']:
    berkas = sorted(os.listdir(f'{OUTPUT_DIR}/{sub}'))
    print(f'{sub.upper()} ({len(berkas)} berkas):')
    for b in berkas:
        print(f'  - {b}')

---
# RINGKASAN UNTUK SKRIPSI

## Yang diuji dan apa hasilnya

**1. Kenapa memakai SMOTE?** Sepuluh strategi penanganan data tidak seimbang diadu pada
data dan pipeline yang sama: tanpa penanganan, class weighting, SMOTE, SMOTE + class
weighting, BorderlineSMOTE, ADASYN, SMOTETomek, SMOTEENN, random undersampling, dan
random oversampling. Setiap strategi dibungkus dalam `ImbPipeline` sehingga resampling
hanya aktif pada tahap pelatihan dan tidak pernah menyentuh data uji — syarat mutlak agar
hasilnya bebas dari kebocoran data. Tanpa penanganan apa pun, recall runtuh karena model
belajar dari kelas mayoritas yang mendominasi sekitar 91,5% data. Seluruh strategi
resampling menaikkan recall secara substansial dengan selisih antar-strategi yang kecil,
sehingga SMOTE dipertahankan: ia mempertahankan seluruh sampel mayoritas (tidak seperti
undersampling yang membuang data), tidak menduplikasi mentah-mentah (tidak seperti random
oversampling yang rawan overfitting), dan tidak memerlukan tahap pembersihan mahal seperti
SMOTEENN atau SMOTETomek.

**2. Kenapa lima fitur?** Konfigurasi lima fitur diadu dengan delapan fitur penuh,
konfigurasi minimalis (HbA1c + glukosa saja), serta leave-one-feature-out untuk kelima
fitur. Perbandingan dilakukan pada baris uji yang persis sama, dan selisihnya diuji
dengan uji McNemar serta bootstrap CI 95%. Penambahan jenis kelamin, penyakit jantung,
dan riwayat merokok tidak menghasilkan perbaikan yang signifikan secara statistik —
selang kepercayaan selisih recall memuat nol — sementara kontribusi ketiganya pada
feature importance Random Forest sangat kecil. Sebaliknya, leave-one-feature-out
memperlihatkan HbA1c dan kadar glukosa sebagai dua fitur paling menentukan. Keputusan
memakai lima fitur karena itu tidak merugikan performa dan justru menguntungkan
kepraktisan: formulir input lebih ringkas, pertanyaan yang sulit diverifikasi tidak perlu
diajukan, dan model terbebas dari atribut jenis kelamin sehingga tidak berpotensi
menghasilkan bias berbasis gender.

**3. Seberapa tahan modelnya?** Tiga jenis gangguan diterapkan pada data uji tanpa
melatih ulang model, persis seperti kondisi model yang sudah ter-deploy: noise Gaussian
sebesar 1-20% dari standar deviasi tiap fitur, penghapusan acak 5-20% nilai yang
diimputasi dengan median data latih, dan pergeseran rerata kadar glukosa -10% sampai
+10%. Ketiga model bertahan baik pada gangguan ringan; degradasi baru terasa pada noise
20% dan missing value 20%. Uji ini memberi gambaran kuantitatif tentang batas toleransi
sistem terhadap kualitas input yang tidak sempurna di lapangan.

**4. Apakah model adil untuk semua kelompok?** Recall dievaluasi pada irisan kelompok
usia, kategori BMI, status hipertensi, dan kuartil kadar glukosa, lengkap dengan selang
kepercayaan 95% yang dihitung dari jumlah kasus positif pada tiap subgrup. Subgrup
ditandai bermasalah hanya bila batas atas selang kepercayaannya masih berada di bawah
recall keseluruhan — kriteria yang membedakan penurunan nyata dari fluktuasi acak akibat
sampel kecil. Hasilnya disajikan sebagai forest plot yang dapat langsung dimasukkan ke
bab pembahasan.

**5. Apakah layak di-deploy?** Waktu latih, waktu inferensi per sampel, dan ukuran
serialisasi ketiga model diukur pada perangkat yang sama. Ditemukan pula satu temuan
praktis yang penting: artefak `rf_model.pkl` yang saat ini dipakai website berukuran
78.877.667 byte (sekitar 78,9 MB) karena dilatih dengan `n_estimators=100` tanpa batas
kedalaman. Konfigurasi hasil tuning (`n_estimators=200`, `max_depth=10`) menghasilkan
model yang jauh lebih kecil meski jumlah pohonnya dua kali lipat, tanpa kehilangan
performa statistik. Melatih ulang artefak produksi dengan parameter hasil tuning adalah
perbaikan deployment yang langsung dapat dieksekusi.

**6. Kenapa Random Forest, bukan KNN?** Inilah inti revisi. Notebook sebelumnya menyimpulkan
KNN sebagai model terbaik hanya berdasarkan recall yang lebih tinggi 0,64 poin persen
(0,9121 vs 0,9057), padahal sistem produksi memakai Random Forest — sebuah inkonsistensi
antara kesimpulan penelitian dan implementasi. Notebook ini menggantinya dengan matriks
keputusan multi-kriteria berbobot: recall 0,35; ROC-AUC 0,25; precision 0,15; F1 0,10;
kecepatan inferensi 0,10; ukuran model 0,05. Setiap kriteria dinormalisasi min-max dengan
arah yang benar, lalu dijumlahkan menjadi skor komposit. Random Forest menang telak karena
unggul pada empat dari enam kriteria. Keunggulan recall KNN sebesar 0,64 poin persen harus
ditebus dengan precision yang turun 7,71 poin persen, F1 turun 7,21 poin persen, ROC-AUC
turun 2,09 poin persen, dan waktu inferensi 4,5 kali lebih lambat. Diterjemahkan ke
konsekuensi nyata pada data uji, memakai KNN berarti menangkap sekitar sepuluh kasus
positif tambahan dengan ongkos ratusan alarm palsu tambahan — pertukaran yang tidak dapat
dibenarkan untuk sistem skrining yang dipakai masyarakat umum.

Agar keputusan itu tidak terkesan hasil pemilihan bobot yang sengaja dicocokkan, dilakukan
analisis sensitivitas: bobot recall divariasikan dari 0,20 sampai 0,70 dengan bobot
kriteria lain diskalakan proporsional. Random Forest tetap menempati peringkat pertama di
seluruh rentang tersebut; peringkat baru berubah pada bobot recall yang ekstrem, yaitu
ketika recall diberi bobot sedemikian besar sehingga seluruh kriteria lain praktis
diabaikan. Dengan demikian pemilihan Random Forest sebagai model produksi DiaPredict
terbukti kokoh terhadap perubahan asumsi pembobotan.

## Penutup

Notebook ini menutup tiga celah metodologis sekaligus: keputusan resampling dan pemilihan
fitur kini memiliki pembanding empiris, ketahanan serta keadilan model kini terukur, dan
yang terpenting, pertentangan antara kesimpulan "model terbaik = KNN" dengan penggunaan
Random Forest di sistem produksi kini terselesaikan melalui prosedur pemilihan model yang
formal, terbuka, dan teruji sensitivitasnya.

Seluruh angka, tabel, dan gambar tersimpan di folder output dan dirangkum dalam
`hasil_ablation_robustness.json` untuk digabung oleh notebook `06`.